# CASCADE — Region Comparison
Compare batch-fitting results across user-selected spatial regions.

**Workflow:**
1. Set `H5_FILE` in the **Configuration** cell and run all cells through *Region Selection*
2. Draw rectangular regions on the image, click **Add Region** after each, then **Done**
3. Run **Peak Extraction** → **Hungarian Matching** → **Class Assignment**
4. Run any visualisation cell in any order

**Visualisations available:**
| Cell | Plot | What it shows |
|---|---|---|
| VIZ 1 | Side-by-side lollipop | Per-peak amplitude, one panel per region |
| VIZ 2 | Overlaid lollipop | All regions on one axis, x-jittered |
| VIZ 3 | Differential lollipop | Δ amplitude between two regions |
| VIZ 4 | Violin by class | Within-region amplitude distribution per class |
| VIZ 5 | Box plots (matched) | Pixel-level amplitude spread for each matched peak |
| VIZ 6 | Spatial heatmaps | Where in the region a class/peak is strongest |
| VIZ 7 | Radar / spider chart | Class-level profile per region |
| VIZ 8 | Grouped bar + error bars | Class-level mean ± std per region |
| VIZ 9 | Prevalence lollipop | Head size = fraction of pixels that have the peak |

In [ ]:
import numpy as np
import h5py
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('widget')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import RectangleSelector, Button

from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import fclusterdata
from scipy.stats import mannwhitneyu

from IPython.display import display, clear_output

print('Imports OK')

In [ ]:
import h5py
import lazy5
from lazy5.inspect import get_datasets, get_attrs_dset
def load_h5_file(filename, galvo=True, data_path=None, nrb_path=None, dark_path=None):
    """Load BCARS hyperspectral data from an HDF5 file.

    Parameters
    ----------
    filename  : path to the .h5 file
    galvo     : unused flag kept for API compatibility
    data_path : explicit dataset path for data. If None, tries known paths then
                auto-detects the first 3-D dataset in the file.
    nrb_path  : explicit dataset path for NRB. If None, tries known paths then
                falls back to a vector of ones.
    dark_path : explicit dataset path for dark frame. If None, tries known paths
                then returns None if not found.

    Returns
    -------
    data, nrb, dark, attrs
      dark is None if no dark frame was found.
    """
    _DATA_CANDIDATES = [
        '/raw_data/hyperspectral_image_0000',
        '/preprocessed_images/medfilter_raw',
        '/preprocessed_images/medfilter_ratio_SVD_KK_PhaseErrorCorrectALS_ScaleErrorCorrectSG',
        '/preprocessed_images/ratio_SVD_KK_PhaseErrorCorrectALS',
        '/preprocessed_images/medfilter_ratio',
        '/preprocessed_images/ratio',
    ]
    _NRB_CANDIDATES = [
        '/preprocessed_images/medfilter_nrb_for_ratio',
        '/preprocessed_images/nrb',
        '/nrb',
    ]
    _DARK_CANDIDATES = [
        '/raw_data/dark_image_pre',
        '/raw_data/dark_image_post',
        '/raw_data/dark',
        '/dark',
    ]

    data, nrb, dark, attrs = None, None, None, None
    found_data_path = None

    with h5py.File(filename, "r") as f:
        all_datasets = get_datasets(f)
        print("Available datasets:", all_datasets)

        # ── Data ──────────────────────────────────────────────────────────────
        if data_path is not None:
            if data_path not in f:
                raise KeyError(
                    f"data_path '{data_path}' not found.\n"
                    f"Available datasets: {all_datasets}"
                )
            found_data_path = data_path
        else:
            for path in _DATA_CANDIDATES:
                if path in f:
                    found_data_path = path
                    break

            if found_data_path is None:
                # Auto-detect: pick the first dataset with 3+ dimensions
                for path in all_datasets:
                    if path in f and len(f[path].shape) >= 3:
                        found_data_path = path
                        print(f"Auto-detected data at: {path} — pass data_path='{path}' to suppress this message")
                        break

            if found_data_path is None:
                raise KeyError(
                    "Could not find a suitable data dataset.\n"
                    f"Available datasets: {all_datasets}\n"
                    "Pass data_path= to specify one explicitly."
                )

        data = np.array(f[found_data_path])
        print(f"Data   loaded from: {found_data_path}  shape: {data.shape}")

        # ── NRB ───────────────────────────────────────────────────────────────
        found_nrb_path = nrb_path
        if found_nrb_path is not None:
            if found_nrb_path not in f:
                raise KeyError(
                    f"nrb_path '{found_nrb_path}' not found.\n"
                    f"Available datasets: {all_datasets}"
                )
        else:
            for path in _NRB_CANDIDATES:
                if path in f:
                    found_nrb_path = path
                    break

        if found_nrb_path is not None:
            nrb = np.array(f[found_nrb_path])
            print(f"NRB    loaded from: {found_nrb_path}  shape: {nrb.shape}")
        else:
            nrb = np.ones(data.shape[-1])
            print(f"NRB not found — using ones  shape: {nrb.shape}")

        # ── Dark ──────────────────────────────────────────────────────────────
        found_dark_path = dark_path
        if found_dark_path is not None:
            if found_dark_path not in f:
                raise KeyError(
                    f"dark_path '{found_dark_path}' not found.\n"
                    f"Available datasets: {all_datasets}"
                )
        else:
            for path in _DARK_CANDIDATES:
                if path in f:
                    found_dark_path = path
                    break

        if found_dark_path is not None:
            dark = np.array(f[found_dark_path])
            print(f"Dark   loaded from: {found_dark_path}  shape: {dark.shape}")
        else:
            print("Dark not found — dark=None")

        # ── Attributes ────────────────────────────────────────────────────────
        attrs = get_attrs_dset(filename, found_data_path)
        print(f"Attrs  loaded from: {found_data_path}")

    return data, nrb, dark, attrs


RAW_PATH = r"/mnt/c/Users/adiering3/Desktop/Misc Snowflakes/preprocessed_medfilter_snowflakemyoG_GFP_09_PROCESS_2026515_13_51_0_818558.h5"
raw_spec,_, _, _ = load_h5_file(RAW_PATH)




In [ ]:
# ── File path ─────────────────────────────────────────────────────────────────
H5_FILE  = "/home/adiering3/projects/CASCADE/fitted_fitted_snowflakemyoG_GFP_09_20260515_212925xsamples_tol=1e-05.h5"    # ← set to your .h5, .npz, or .pt path
PT_SHAPE = None   # for .pt files: set to (H, W) to specify spatial dims, or None to auto-detect

# ── Detection thresholds ─────────────────────────────────────────────────────
AMP_THRESHOLD          = 0.0001   # peaks below this amplitude are ignored
MIN_AMP_DISPLAY      = AMP_THRESHOLD       # minimum amplitude for a peak to be shown in the lollipop plot
CENTER_MATCH_THRESHOLD = 7.0   # cm⁻¹ — peaks closer than this are the same peak

# ── Peak classes ─────────────────────────────────────────────────────────────
# Each top-level key is a class name (e.g. 'nucleotides').
# Each sub-dict maps a human-readable peak label to either:
#   - a single wavenumber (int/float): matched within ±CENTER_MATCH_THRESHOLD
#   - a (lo, hi) tuple: matched by inclusive range
PEAK_CLASSES = {
    'nucleotides': {
        'G, A':             1578,
        'G, A CH def':      (1420, 1480),
        'A, G CH def':      1342,
        'G CH def':         1320,
        'T, A':             (1220, 1284),
        'PO2- str':         (1060, 1095),
        'O-P-O asym str':   828,
        'O-P-O str RNA':    811,
        'O-P-O str DNA':    788,
        'U, C, T ring br':  782,
        'A ring br':        729,
        'T, G ring br':     667,
    },
    'proteins': {
        'Amide I':               (1655, 1680),
        'C=C Tyr, Trp':          1617,
        'C=C Phe, Tyr':          1607,
        'CH def':                (1420, 1480),
        'CH def (1342)':         1342,
        'CH def (1320)':         1320,
        'Amide III':             (1220, 1284),
        'C-C6H5 str Phe, Trp':  1209,
        'C-H bend Tyr':          1176,
        'C-C/C-N str':           1158,
        'C-N str':               1128,
        'C-H in-plane Phe':      1033,
        'Sym. Ring br Phe':      1005,
        'C-C BK str beta-sheet': 980,
        'C-C BK str alpha-helix':937,
        'Ring br Tyr':           854,
        'Ring br Trp':           760,
        'C-C twist Tyr':         645,
        'C-C twist Phe':         621,
    },
    'lipids': {
        'C=O ester':        1736,
        'C=C str':          (1655, 1680),
        'CH def':           (1420, 1480),
        'CH2 twist':        1301,
        '=CH bend':         (1220, 1284),
        '=CH bend (980)':   980,
        'Chain C-C str':    (1060, 1095),
        'CN+(CH3)3 str':    717,
    },
    'carbohydrates': {
        'CH def':           (1420, 1480),
        'C-O str':          1128,
        'C-O, C-C str':     (1060, 1095),
        'C-O-C glycos':     937,
        'C-O-C ring':       877,
    },
}

REGION_COLORS = [
    'tab:blue', 'tab:orange', 'tab:green', 'tab:red',
    'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray',
]

# ── Font sizes ────────────────────────────────────────────────────────────────
# Edit these values to resize text across all visualisation cells at once.
FONT_SIZES = {
    'title':      24,   # figure suptitle
    'subtitle':   20,   # per-panel / subplot titles
    'axis_label': 18,   # x / y axis labels
    'tick_label':  18,   # tick-mark labels
    'legend':      12,   # legend text
    'annotation':  12,   # significance stars, small text
}

from matplotlib.colors import LinearSegmentedColormap

def truncated_cmap(cmap_name, minval=0.0, maxval=0.75, n=256):
    base = plt.cm.get_cmap(cmap_name)
    colors = base(np.linspace(minval, maxval, n))
    # replace the low end with a fade from white
    fade_len = n // 4
    for i in range(fade_len):
        t = i / fade_len  # 0 → 1
        colors[i] = (1-t) * np.array([1,1,1,1]) + t * colors[i]
    return LinearSegmentedColormap.from_list(
        f'{cmap_name}_white', colors, N=n)

plasma_rp = truncated_cmap('plasma_r', minval=0.0, maxval=0.75)

In [ ]:
def _load_h5(path):
    with h5py.File(path, 'r') as f:
        g   = f['preprocessed_images']
        pp  = g['peak_params'][:]
        xa  = g['x_axis'][:]
        mdl = g['model'][:] if 'model' in g else None
        raw = g['raw'][:]   if 'raw'   in g else None
    return pp, xa, mdl, raw

def _load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d['peak_params'], d['x_axis'], d.get('model'), d.get('raw')

def _load_pt(path, spatial_shape=None):
    import torch
    d = torch.load(path, map_location='cpu', weights_only=False)
    def _to_np(v):
        if v is None:
            return None
        return v.numpy() if hasattr(v, 'numpy') else np.asarray(v)
    params = _to_np(d['params_all'])     # (N, P*4)
    xa     = _to_np(d['x_axis'])         # (n_wav,)
    model  = _to_np(d.get('model_all'))  # (N, n_wav) or None
    _raw_key = 'spectra_np' if 'spectra_np' in d else ('spec_d' if 'spec_d' in d else None)
    raw    = _to_np(d[_raw_key]) if _raw_key else None
    N = params.shape[0]
    if spatial_shape is not None:
        H_pt, W_pt = spatial_shape
    else:
        sq = int(round(N ** 0.5))
        H_pt, W_pt = (sq, sq) if sq * sq == N else (N, 1)
    params = params.reshape(H_pt, W_pt, -1)
    if model is not None:
        model = model.reshape(H_pt, W_pt, -1)
    if raw is not None:
        raw = raw.reshape(H_pt, W_pt, -1)
    return params, xa, model, raw

ext = H5_FILE.rsplit('.', 1)[-1].lower()
if ext in ('h5', 'hdf5'):
    peak_params, x_axis, model, raw = _load_h5(H5_FILE)
elif ext == 'pt':
    peak_params, x_axis, model, raw = _load_pt(H5_FILE, spatial_shape=PT_SHAPE)
else:
    peak_params, x_axis, model, raw = _load_npz(H5_FILE)

H, W, P   = peak_params.shape
max_peaks = P // 4
peaks_4d  = peak_params.reshape(H, W, max_peaks, 4)
# peaks_4d[y, x, k, 0] = amplitude  (0–1, normalised)
# peaks_4d[y, x, k, 1] = center     (cm⁻¹)
# peaks_4d[y, x, k, 2] = sigma      (cm⁻¹, Gaussian half-width)
# peaks_4d[y, x, k, 3] = gamma      (cm⁻¹, Lorentzian half-width)

if model is not None:
    display_img = model.mean(axis=2)
elif raw is not None:
    display_img = raw.astype(float).mean(axis=2)
else:
    display_img = peaks_4d[:, :, :, 0].max(axis=2)

wn_lo, wn_hi = float(x_axis.min()), float(x_axis.max())
print(f'Loaded {H}×{W} pixels  |  max_peaks={max_peaks}  |  WN {wn_lo:.0f}–{wn_hi:.0f} cm⁻¹')

In [ ]:
print(f'model    min/max: {model.min():.6f}  {model.max():.6f}')
print(f'peaks_4d min/max: {peaks_4d[...,0].min():.6f}  {peaks_4d[...,0].max():.6f}')

In [ ]:
# compute norm factor from model and normalize both model and peaks_4d
NORMALIZATION = 'peak'
NORM_PEAK_WN  = 1004.0
def normalize_peaks_4d(peaks_4d, mode='power', ref_wn=None):
    amps = peaks_4d[..., 0].copy()

    if mode == 'max':
        norm = amps.max(axis=2, keepdims=True)
    elif mode == 'power':
        norm = np.sqrt((amps**2).mean(axis=2, keepdims=True))
    elif mode == 'peak':
        ctrs = peaks_4d[..., 1]
        dist = np.abs(ctrs - ref_wn)
        idx  = dist.argmin(axis=2)
        norm = amps[np.arange(amps.shape[0])[:, None],
                    np.arange(amps.shape[1])[None, :],
                    idx][..., np.newaxis]
    else:
        raise ValueError(f'Unknown normalization mode: {mode}')

    norm = np.where(norm < 1e-12, 1.0, norm)
    out  = peaks_4d.copy()
    out[..., 0] = amps / norm
    return out, norm


# normalize model and peaks_4d with the same factor
# model    = model    / norm_factor                     # (H, W, n_wn)
peaks_4d_norm = normalize_peaks_4d(
    peaks_4d, mode=NORMALIZATION, ref_wn=NORM_PEAK_WN)[0]

print(f'Normalized — mode={NORMALIZATION}')
print(f'model    min/max: {model.min():.6f}  {model.max():.6f}')
print(f'peaks_4d min/max: {peaks_4d_norm[...,0].min():.6f}  {peaks_4d_norm[...,0].max():.6f}')

In [ ]:
plt.plot(x_axis, model[150,220,:])
plt.show()

## Region Selection
1. Click and drag on the image to draw a rectangle
2. Click **Add Region** to confirm it
3. Repeat for each region you want to compare
4. Click **Done** when finished — then run the next cell

In [ ]:

fig_sel, ax_img = plt.subplots(figsize=(8, 6))
im = ax_img.imshow(display_img, cmap='inferno', aspect='auto')
ax_img.set_title('Click polygon vertices — Add Region after each, then Done')
plt.colorbar(im, ax=ax_img, fraction=0.03, label='Mean Spectral Intensity')
plt.tight_layout()

plt.show()

In [ ]:
from matplotlib.widgets import PolygonSelector
from matplotlib.patches import Polygon
import numpy as np

regions     = []
_pending    = [None]
_live_patch = [None]

fig_sel, ax_img = plt.subplots(figsize=(8, 6))
im = ax_img.imshow(display_img, cmap='inferno', aspect='auto')
ax_img.set_title('Click polygon vertices — Add Region after each, then Done')
plt.colorbar(im, ax=ax_img, fraction=0.03, label='Intensity')
import json, pathlib

REGIONS_FILE = pathlib.Path('regions.json')   # ← change path if you like

# ── persistence helpers ──────────────────────────────────────────────────────

def save_regions(path=REGIONS_FILE):
    data = [
        {'name': r['name'],
         'verts': r['verts'].tolist(),   # ndarray → plain list
         'color': r['color']}
        for r in regions
    ]
    path.write_text(json.dumps(data, indent=2))
    print(f'Saved {len(data)} region(s) → {path}')

def load_regions(path=REGIONS_FILE):
    """Call this BEFORE showing the figure to restore a previous session."""
    if not path.exists():
        print('No saved regions found.')
        return
    data = json.loads(path.read_text())
    regions.clear()
    for ax_patch in list(ax_img.patches):
        ax_patch.remove()
    for item in data:
        verts = np.array(item['verts'])
        color = item['color']
        patch = Polygon(verts, closed=True,
                        linewidth=2, edgecolor=color,
                        facecolor=color, alpha=0.2, linestyle='-')
        ax_img.add_patch(patch)
        regions.append(dict(name=item['name'], verts=verts, color=color))
    fig_sel.canvas.draw_idle()
    print(f'Loaded {len(regions)} region(s) from {path}')
    
def _on_poly_select(verts):
    verts = np.array(verts)  # shape (N, 2), columns = [x, y]
    color = REGION_COLORS[len(regions) % len(REGION_COLORS)]
    if _live_patch[0] is not None:
        try:
            _live_patch[0].remove()
        except Exception:
            pass
    patch = Polygon(verts, closed=True,
                    linewidth=2, edgecolor=color,
                    facecolor=color, alpha=0.2, linestyle='--')
    ax_img.add_patch(patch)
    _live_patch[0] = patch
    _pending[0] = dict(verts=verts, color=color)
    fig_sel.canvas.draw_idle()

_ps = PolygonSelector(ax_img, _on_poly_select, useblit=True)

def _btn_add(_):
    if _pending[0] is None:
        print('Draw a polygon first, then click Add Region.')
        return
    name = f'Region {len(regions) + 1}'
    reg  = dict(name=name, **_pending[0])
    regions.append(reg)
    _pending[0] = None
    if _live_patch[0] is not None:
        _live_patch[0].set_linestyle('-')
        _live_patch[0] = None
    # reset selector so a new polygon can be drawn
    _ps.clear()
    fig_sel.canvas.draw_idle()
    print(f'Added {name}  verts={reg["verts"].tolist()}')

def _btn_clear(_):
    regions.clear()
    for p in list(ax_img.patches):
        p.remove()
    _pending[0] = None
    _live_patch[0] = None
    _ps.clear()
    fig_sel.canvas.draw_idle()
    print('Cleared all regions.')

def _btn_done(_):
    if _pending[0] is not None:
        _btn_add(None)
    print(f'{len(regions)} region(s) confirmed — run the next cell.')
# ── button callback ──────────────────────────────────────────────────────────

def _btn_save(_):
    save_regions()

# ── layout: shift existing buttons right to make room ───────────────────────
ax_add   = plt.axes([0.01, 0.01, 0.18, 0.055])
ax_clear = plt.axes([0.21, 0.01, 0.18, 0.055])
ax_save  = plt.axes([0.41, 0.01, 0.18, 0.055])
ax_done  = plt.axes([0.61, 0.01, 0.18, 0.055])

btn_add   = Button(ax_add,   'Add Region')
btn_clear = Button(ax_clear, 'Clear All')
btn_save  = Button(ax_save,  'Save')
btn_done  = Button(ax_done,  'Done')

btn_add.on_clicked(_btn_add)
btn_clear.on_clicked(_btn_clear)
btn_save.on_clicked(_btn_save)
btn_done.on_clicked(_btn_done)

# ── restore previous session (runs once when the cell executes) ──────────────
load_regions()

plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()


In [ ]:
from matplotlib.path import Path

def _poly_slices(reg):
    """Returns sa, sc, poly_mask (broadcast-ready) for a polygon region."""
    verts = reg['verts']
    c0, r0 = verts.min(axis=0).astype(int)
    c1, r1 = verts.max(axis=0).astype(int)
    c0, r0 = max(0, c0), max(0, r0)
    c1, r1 = min(W-1, c1), min(H-1, r1)
    sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
    sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
    rH, rW = sa.shape[:2]
    grid_cols, grid_rows = np.meshgrid(
        np.arange(c0, c1+1), np.arange(r0, r1+1))
    grid_pts = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
    poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
    poly_mask    = poly_mask_2d[:, :, np.newaxis]
    return sa, sc, poly_mask, poly_mask_2d

## Peak Extraction and Matching
Run cells in order: **Extract → Match → Classify**

In [ ]:

def _get_region_peaks(reg):
    verts = reg['verts']  # (N, 2) array of [x, y] = [col, row]
    
    # bounding box to limit the search area
    c0, r0 = verts.min(axis=0).astype(int)
    c1, r1 = verts.max(axis=0).astype(int)
    c0, r0 = max(0, c0), max(0, r0)
    c1, r1 = min(W-1, c1), min(H-1, r1)

    sub  = peaks_4d[r0:r1+1, c0:c1+1, :, :]   # (rH, rW, P, 4)
    amps = sub[..., 0]
    ctrs = sub[..., 1]
    sigs = sub[..., 2]
    gams = sub[..., 3]

    # build polygon mask over the bounding box
    rH, rW = sub.shape[:2]
    grid_cols, grid_rows = np.meshgrid(
        np.arange(c0, c1+1), np.arange(r0, r1+1))          # (rH, rW)
    grid_pts = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
    poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)

    # broadcast polygon mask over peaks dimension and apply amp threshold
    poly_mask = poly_mask_2d[:, :, np.newaxis]               # (rH, rW, 1)
    amp_mask  = amps > AMP_THRESHOLD                         # (rH, rW, P)
    mask      = poly_mask & amp_mask

    n_px = int(poly_mask_2d.sum())
    return dict(
        amp=amps[mask], center=ctrs[mask],
        sigma=sigs[mask], gamma=gams[mask],
        pixel_amps=amps, pixel_ctrs=ctrs, pixel_mask=mask,
        n_pixels=n_px, sub_shape=sub.shape[:2],
    )
def _cluster_peaks(peaks_dict, threshold=CENTER_MATCH_THRESHOLD):
    centers = peaks_dict['center']
    amps    = peaks_dict['amp']
    sigmas  = peaks_dict['sigma']
    gammas  = peaks_dict['gamma']
    n_px    = peaks_dict['n_pixels']

    if len(centers) == 0:
        return []
    if len(centers) == 1:
        idx = np.array([0])
        return [dict(
            mean_center = float(centers[0]),
            std_center  = 0.0,
            mean_amp    = float(amps[0]),
            std_amp     = 0.0,
            mean_sigma  = float(sigmas[0]),
            mean_gamma  = float(gammas[0]),
            mean_fwhm   = float(2.35482 * sigmas[0] + 2.0 * gammas[0]),
            prevalence  = 1.0 / n_px,
            n_obs       = 1,
            cls         = None,
        )]

    # build a density histogram with bin width = threshold/2
    bin_width = threshold / 2
    wn_min, wn_max = centers.min(), centers.max()
    n_bins    = max(1, int(np.ceil((wn_max - wn_min) / bin_width)))
    bin_edges = np.linspace(wn_min, wn_max + 1e-6, n_bins + 1)
    counts, _ = np.histogram(centers, bins=bin_edges)

    # find local maxima in the histogram — each is a consensus peak
    from scipy.signal import find_peaks
    # min_distance in bins = threshold / bin_width = 2 bins
    peak_bins, _ = find_peaks(counts, distance=max(1, int(threshold / bin_width)))

    # if no local maxima found (e.g. flat histogram), fall back to global max
    if len(peak_bins) == 0:
        peak_bins = np.array([counts.argmax()])

    # assign each observation to its nearest peak bin
    bin_centers  = (bin_edges[:-1] + bin_edges[1:]) / 2
    obs_bins     = np.digitize(centers, bin_edges) - 1
    obs_bins     = np.clip(obs_bins, 0, n_bins - 1)
    peak_bin_arr = np.array(peak_bins)

    # nearest peak bin for each observation
    dists       = np.abs(obs_bins[:, None] - peak_bin_arr[None, :])
    assignments = peak_bin_arr[dists.argmin(axis=1)]

    result = []
    for pb in peak_bins:
        mask = assignments == pb
        if not mask.any():
            continue
        c, a, s, g = centers[mask], amps[mask], sigmas[mask], gammas[mask]
        fwhm_g = 2.35482 * s.mean()
        fwhm_l = 2.0     * g.mean()
        result.append(dict(
            mean_center = float(c.mean()),
            std_center  = float(c.std()),
            mean_amp    = float(a.mean()),
            std_amp     = float(a.std()),
            mean_sigma  = float(s.mean()),
            mean_gamma  = float(g.mean()),
            mean_fwhm   = float(fwhm_g + fwhm_l),
            prevalence  = float(mask.sum()) / n_px,
            n_obs       = int(mask.sum()),
            cls         = None,
        ))

    result.sort(key=lambda d: d['mean_center'])
    return result
assert len(regions) >= 2, 'Select at least 2 regions before running this cell.'

for reg in regions:
    pd_ = _get_region_peaks(reg)
    reg['peaks_dict'] = pd_
    reg['consensus']  = _cluster_peaks(pd_)
    name = reg['name']
    n_c  = len(reg['consensus'])
    n_o  = pd_['amp'].size
    n_px = pd_['n_pixels']
    print(f'{name}: {n_c} consensus peaks from {n_o} observations ({n_px} pixels)')

In [ ]:
def match_peaks(reg_a, reg_b, threshold=CENTER_MATCH_THRESHOLD):
    """Hungarian matching of consensus peaks between two regions."""
    ca_list = reg_a['consensus']
    cb_list = reg_b['consensus']
    if not ca_list or not cb_list:
        return []
    ca   = np.array([p['mean_center'] for p in ca_list])
    cb   = np.array([p['mean_center'] for p in cb_list])
    cost = np.abs(ca[:, None] - cb[None, :])
    pen  = np.where(cost <= threshold, cost, 1e9)
    ri, ci = linear_sum_assignment(pen)
    matches = []
    for r, c in zip(ri, ci):
        if cost[r, c] <= threshold:
            matches.append(dict(
                idx_a=int(r), idx_b=int(c),
                peak_a=ca_list[r], peak_b=cb_list[c],
                center_a=float(ca[r]), center_b=float(cb[c]),
                delta_center=float(cb[c] - ca[r]),
                cost=float(cost[r, c]),
            ))
    return matches

ref = regions[0]
for reg in regions[1:]:
    reg['matches_vs_ref'] = match_peaks(ref, reg)
    n_ref   = len(ref['consensus'])
    n_other = len(reg['consensus'])
    n_m     = len(reg['matches_vs_ref'])
    ref_n   = ref['name']
    other_n = reg['name']
    print(f'{ref_n} ({n_ref} peaks)  ↔  {other_n} ({n_other} peaks):  {n_m} matched')

In [ ]:
def _class_wn_mask(sa, sc, class_peaks):
    combined = np.zeros(sa.shape, dtype=bool)
    for peak_val in class_peaks.values():
        if isinstance(peak_val, tuple):
            lo, hi = peak_val
            combined |= (sa > AMP_THRESHOLD) & (sc >= lo) & (sc <= hi)
        else:
            combined |= (sa > AMP_THRESHOLD) & (np.abs(sc - peak_val) <= CENTER_MATCH_THRESHOLD)
    return combined

def _peak_in_class(center, class_peaks):
    for peak_val in class_peaks.values():
        if isinstance(peak_val, tuple):
            lo, hi = peak_val
            if lo <= center <= hi:
                return True
        else:
            if abs(center - peak_val) <= CENTER_MATCH_THRESHOLD:
                return True
    return False

def _assign_classes(consensus, class_def=PEAK_CLASSES):
    for pk in consensus:
        pk['cls'] = None
        for cls, class_peaks in class_def.items():
            if _peak_in_class(pk['mean_center'], class_peaks):
                pk['cls'] = cls
                break

def _class_summary(reg, class_def=PEAK_CLASSES):
    verts = reg['verts']
    c0, r0 = verts.min(axis=0).astype(int)
    c1, r1 = verts.max(axis=0).astype(int)
    c0, r0 = max(0, c0), max(0, r0)
    c1, r1 = min(W-1, c1), min(H-1, r1)

    sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
    sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

    # polygon mask broadcast over peaks dimension
    rH, rW = sa.shape[:2]
    grid_cols, grid_rows = np.meshgrid(
        np.arange(c0, c1+1), np.arange(r0, r1+1))
    grid_pts = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
    poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
    poly_mask    = poly_mask_2d[:, :, np.newaxis]            # (rH, rW, 1)

    n_px = reg['peaks_dict']['n_pixels']
    summary = {}
    for cls, class_peaks in class_def.items():
        mask = _class_wn_mask(sa, sc, class_peaks) & poly_mask
        vals = sa[mask]
        members = [p for p in reg['consensus'] if p.get('cls') == cls]
        summary[cls] = dict(
            n_peaks    = len(members),
            total_amp  = float(vals.sum()  / n_px) if len(vals) > 0 else 0.0,
            mean_amp   = float(vals.mean())         if len(vals) > 0 else 0.0,
            std_amp    = float(vals.std())          if len(vals) > 0 else 0.0,
            max_amp    = float(vals.max())          if len(vals) > 0 else 0.0,
            prevalence = float((mask.any(axis=2) & poly_mask_2d).mean()),
        )
    return summary
# ── Significance helpers ───────────────────────────────────────────────────────────────────────────────


In [ ]:
def _sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return None

from scipy.stats import ks_2samp

from scipy.stats import ttest_ind

def _pairwise_sig(data_list, threshold=0.05, min_cohens_d=0.5):
    from itertools import combinations
    results = []
    for (i, d1), (j, d2) in combinations(enumerate(data_list), 2):
        d1 = np.asarray(d1).ravel()
        d2 = np.asarray(d2).ravel()
        if len(d1) < 3 or len(d2) < 3:
            continue
        if np.all(d1 == d1[0]) and np.all(d2 == d2[0]):
            continue
        try:
            _, p = ttest_ind(d1, d2, equal_var=False)
        except Exception:
            continue
        pooled_std = np.sqrt((d1.std()**2 + d2.std()**2) / 2)
        d = abs(d1.mean() - d2.mean()) / pooled_std if pooled_std > 0 else 0.0
        lbl = _sig_label(p)
        if lbl is not None and d >= min_cohens_d:
            results.append((i, j, lbl))
    return results

def _draw_sig_brackets(ax, pairs, x_positions, y_top, dy, fontsize=8):
    """
    Draw significance brackets.
    pairs       : [(i, j, label), ...] — i, j are indices into x_positions
    x_positions : list of x coords, one per group being compared
    y_top       : starting y for the lowest bracket
    dy          : vertical step between stacked brackets
    """
    if not pairs:
        return
    tick_h = dy * 0.4
    y = y_top
    for i, j, lbl in pairs:
        x1, x2 = x_positions[i], x_positions[j]
        ax.plot([x1, x1, x2, x2], [y, y + tick_h, y + tick_h, y],
                lw=0.9, color='black', clip_on=False)
        ax.text((x1 + x2) / 2, y + tick_h, lbl,
                ha='center', va='bottom', fontsize=fontsize,
                clip_on=False, fontweight='bold')
        y += dy
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, max(hi, y + tick_h * 2))

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────────

for reg in regions:
    _assign_classes(reg['consensus'])
    reg['class_summary'] = _class_summary(reg)

print('Class assignment complete.  Summary (mean_amp | prevalence):')
col_w = 22
hdr = 'Class'.ljust(36) + ''.join(r['name'].rjust(col_w) for r in regions)
print(hdr)
print('-' * len(hdr))
for cls in PEAK_CLASSES:
    row = cls.ljust(36)
    for reg in regions:
        v   = reg['class_summary'][cls]
        row += f'{v["mean_amp"]:.4f}/{v["prevalence"]:.1%}'.rjust(col_w)
    print(row)


## Visualisations
Each cell below is independent — run them in any order.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 1 — Side-by-side lollipop
#  One panel per region; head colour = peak class; error bar = std_amp
#  Asterisks (*/**/***)  mark peaks significantly different from the
#  reference region (Mann-Whitney U on pixel-level amplitudes)
# ═══════════════════════════════════════════════════════════════════

def _cls_palette():
    keys   = list(PEAK_CLASSES.keys())
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(keys), 1)))
    return {k: colors[i] for i, k in enumerate(keys)}
def plot_lollipop(regions, metric='mean_amp', figsize=None):
    if figsize is None:
        figsize = (5 * len(regions), 5)
    palette  = _cls_palette()
    all_vals = [p[metric] for r in regions for p in r['consensus']]
    y_max    = max(all_vals) if all_vals else 1.0

    fig, axes = plt.subplots(1, len(regions), figsize=figsize, sharey=True)
    if len(regions) == 1:
        axes = [axes]

    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    ref = regions[0]

    sig_maps = [{} for _ in regions]

    for ri, reg in enumerate(regions):
        if ri == 0:
            for other in regions[1:]:
                for m in other.get('matches_vs_ref', []):
                    d1 = _px_amps(ref, m['center_a'])
                    d2 = _px_amps(other, m['center_b'])
                    pairs = _pairwise_sig([d1, d2])
                    if pairs:
                        wn_key = round(m['center_a'], 1)
                        existing = sig_maps[0].get(wn_key, '*')
                        new_lbl  = pairs[0][2]
                        sig_maps[0][wn_key] = new_lbl if len(new_lbl) >= len(existing) else existing
        else:
            for m in reg.get('matches_vs_ref', []):
                d1 = _px_amps(ref, m['center_a'])
                d2 = _px_amps(reg, m['center_b'])
                pairs = _pairwise_sig([d1, d2])
                if pairs:
                    wn_key = round(m['center_b'], 1)
                    sig_maps[ri][wn_key] = pairs[0][2]

    for ri, (ax, reg) in enumerate(zip(axes, regions)):
        for pk in reg['consensus']:
            x     = pk['mean_center']
            y     = pk[metric]
            color = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.5, alpha=0.65)
            ax.scatter(x, y, s=55, color=color, zorder=5,
                       edgecolors='k', linewidths=0.4)
            if pk['std_amp'] > 0:
                ax.errorbar(x, y, yerr=pk['std_amp'], fmt='none',
                            ecolor='gray', elinewidth=0.8, capsize=2)
            lbl = sig_maps[ri].get(round(x, 1))
            if lbl:
                ax.text(x, y + pk.get('std_amp', 0) + y_max * 0.03,
                        lbl, ha='center', va='bottom',
                        fontsize=FONT_SIZES['annotation'], fontweight='bold')
        ax.set_xlim(wn_lo - 15, wn_hi + 15)
        ax.set_ylim(0, y_max * 1.35)
        ax.set_title(reg['name'], color=reg['color'], fontweight='bold',
                     fontsize=FONT_SIZES['subtitle'])
        ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
        ax.tick_params(labelsize=FONT_SIZES['tick_label'])
        ax.spines[['top', 'right']].set_visible(False)

    axes[0].set_ylabel(metric.replace('_', ' ').title(),
                       fontsize=FONT_SIZES['axis_label'])
    handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    fig.legend(handles=handles, loc='upper right', fontsize=FONT_SIZES['legend'],
               title='Peak Class', framealpha=0.85)
    metric_lbl = metric.replace('_', ' ').title()
    fig.suptitle(f'Lollipop — {metric_lbl}', fontsize=FONT_SIZES['title'])
    plt.tight_layout()
    plt.show()

plot_lollipop(regions, metric='mean_amp')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 2 — Overlaid lollipop
#  All regions on one axis; peaks are x-jittered for readability
# ═══════════════════════════════════════════════════════════════════

def plot_lollipop_overlay(regions, metric='mean_amp', figsize=(12, 5)):
    palette = _cls_palette()
    n      = len(regions)
    jitter = np.linspace(-2.5, 2.5, n) if n > 1 else [0.0]

    fig, ax = plt.subplots(figsize=figsize)
    for reg, jit in zip(regions, jitter):
        for pk in reg['consensus']:
            x     = pk['mean_center'] + jit
            y     = pk[metric]
            color = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.5, alpha=0.6)
            ax.scatter(x, y, s=48, color=color, zorder=5,
                       edgecolors=reg['color'], linewidths=0.8)

    reg_handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    leg1 = ax.legend(handles=reg_handles, loc='upper left',
                     fontsize=FONT_SIZES['legend'], title='Region')
    ax.add_artist(leg1)
    ax.legend(handles=cls_handles, loc='upper right',
              fontsize=FONT_SIZES['legend'], title='Class')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=FONT_SIZES['axis_label'])
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_title(f'Overlaid Lollipop — {metric_lbl}', fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_lollipop_overlay(regions, metric='mean_amp')

In [ ]:
def _mean_spectrum(reg):
        cube  = model
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)
        sub  = cube[r0:r1+1, c0:c1+1, :]
        rH, rW = sub.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        return sub[poly_mask_2d, :].mean(axis=0)


In [ ]:
MIN_AMP_DISPLAY   = 0.0002    # hide lollipops below this mean_amp
SIG_ONLY          = True   # if True, only plot peaks with a sig difference
SHOW_NONSIG_STEMS = False    # if False, hide stems for non-sig peaks entirely


def plot_lollipop_overlay(regions, metric='mean_amp', figsize=(14, 5)):
    palette = _cls_palette()
    n      = len(regions)
    jitter = np.linspace(-2.5, 2.5, n) if n > 1 else [0.0]
    jitter_map = {id(reg): jit for reg, jit in zip(regions, jitter)}

    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    # collect significant matched pairs: (x_a, x_b, y_a, y_b, label)
    ref = regions[0]
    sig_pairs = []
    for other in regions[1:]:
        for m in other.get('matches_vs_ref', []):
            d1 = _px_amps(ref, m['center_a'])
            d2 = _px_amps(other, m['center_b'])
            pairs = _pairwise_sig([d1, d2])
            if not pairs:
                continue
            lbl  = pairs[0][2]
            y_a  = m['peak_a'].get(metric, 0)
            y_b  = m['peak_b'].get(metric, 0)
            # skip if either peak is below amplitude threshold
            if y_a < MIN_AMP_DISPLAY or y_b < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                x_a  = m['center_a'] + jitter_map[id(ref)],
                x_b  = m['center_b'] + jitter_map[id(other)],
                y_a  = y_a,
                y_b  = y_b,
                lbl  = lbl,
                pk_a = m['peak_a'],
                pk_b = m['peak_b'],
                reg_a = ref,
                reg_b = other,
            ))

    # decide which peaks to show
    visible_keys = set()   # (region id, round(wn, 1))
    for sp in sig_pairs:
        visible_keys.add((id(sp['reg_a']), round(sp['pk_a']['mean_center'], 1)))
        visible_keys.add((id(sp['reg_b']), round(sp['pk_b']['mean_center'], 1)))

    fig, ax = plt.subplots(figsize=figsize)
    
        
    for ri, (reg, jit) in enumerate(zip(regions, jitter)):
        for pk in reg['consensus']:
            y   = pk[metric]
            wn  = pk['mean_center']
            key = (id(reg), round(wn, 1))
            if SIG_ONLY and key not in visible_keys:
                continue
            if y < MIN_AMP_DISPLAY:
                continue
            x     = wn + jit
            color = palette.get(pk.get('cls'), 'lightgray')
            is_sig = key in visible_keys
            alpha  = 0.8 if is_sig else 0.2
            lw     = 2.0 if is_sig else 0.8
            ax.vlines(x, 0, y, color=reg['color'], lw=lw, alpha=alpha)
            ax.scatter(x, y, s=55 if is_sig else 25,
                       color=color, zorder=5,
                       edgecolors=reg['color'] if is_sig else 'none',
                       linewidths=0.8, alpha=alpha)

    ax.set_ylim(bottom=0)
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
    tick_h  = y_range * 0.03
    dy_step = y_range * 0.06

    # sort pairs by x midpoint so brackets stack cleanly left to right
    sig_pairs.sort(key=lambda sp: (sp['x_a'] + sp['x_b']) / 2)

    # stack brackets — track highest y used per x region to avoid overlap
    bracket_tops = {}
    for sp in sig_pairs:
        x1, x2   = sp['x_a'], sp['x_b']
        y_base   = max(sp['y_a'], sp['y_b'])
        x_mid    = (x1 + x2) / 2
        x_slot   = round(x_mid, 0)

        # find a y level that doesn't overlap existing brackets at this x
        y_bracket = y_base + tick_h * 2
        while bracket_tops.get(x_slot, 0) >= y_bracket:
            y_bracket += dy_step
        bracket_tops[x_slot] = y_bracket + tick_h

        ax.plot([x1, x1, x2, x2],
                [y_bracket, y_bracket + tick_h, y_bracket + tick_h, y_bracket],
                lw=0.9, color='black', clip_on=False)
        ax.text(x_mid, y_bracket + tick_h, sp['lbl'],
                ha='center', va='bottom', fontsize=8,
                fontweight='bold', color='black', clip_on=False)

    # extend ylim to fit brackets
    if bracket_tops:
        ax.set_ylim(0, max(bracket_tops.values()) + dy_step * 2)

    reg_handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    leg1 = ax.legend(handles=reg_handles, loc='upper left',
                     fontsize=FONT_SIZES['legend'], title='Region')
    ax.add_artist(leg1)
    ax.legend(handles=cls_handles, loc='upper right',
              fontsize=FONT_SIZES['legend'], title='Class')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=FONT_SIZES['axis_label'])
    ax.set_title(f'Overlaid Lollipop — {metric.replace("_", " ").title()}',
                 fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_lollipop_overlay(regions, metric='mean_amp')

In [ ]:
SHOW_RAW_SPECTRA   = False    # overlay mean raw spectrum for each region
SHOW_ALL_PEAKS     = False   # True = all fitted peaks, False = sig/unmatched only
SHOW_MODEL_SPECTRA = False   # use fitted model cube instead of raw for background

from scipy.special import wofz

def voigt_peak(x, center, amp, sigma, gamma=5.0, Real=False):
    if wofz is None:
        raise ImportError("scipy.special.wofz is required for voigt_peak")

    z = ((x - center) + 1j * gamma) / (sigma * np.sqrt(2))
    w = wofz(z)

    if Real:
        # Dispersive (Hilbert-transform-like) profile.
        # Normalize so `amp` corresponds to the peak absolute amplitude (|y| max).
        profile = np.imag(w) / (sigma * np.sqrt(2 * np.pi))
        peak_ref = float(np.max(np.abs(profile)))
        return amp * profile / (peak_ref + 1e-12)

    # Absorptive Voigt profile.
    # Normalize so `amp` corresponds to the peak height (y max).
    profile = np.real(w) / (sigma * np.sqrt(2 * np.pi))
    peak_ref = float(np.max(profile))
    return amp * profile / (peak_ref + 1e-12)


def plot_spectra_overlay(regions, figsize=(14, 6)):
    palette    = _cls_palette()
    n          = len(regions)
    jitter_map = {id(reg): jit for reg, jit
                  in zip(regions, np.linspace(-2.5, 2.5, n) if n > 1 else [0.0])}

    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    def _mean_spectrum(reg):
        cube = model if (SHOW_MODEL_SPECTRA and model is not None) else raw
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)
        sub  = cube[r0:r1+1, c0:c1+1, :]
        rH, rW = sub.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        pixels = sub[poly_mask_2d, :]
        return pixels.mean(axis=0)

    def _region_peak_curve(pk):
        return np.array(voigt_peak(
            x_axis,
            pk['mean_center'], pk['mean_amp'],
            pk['mean_sigma'],  pk['mean_gamma']))

    # ── collect sig pairs + unmatched ────────────────────────────
    ref       = regions[0]
    sig_pairs = []
    for other in regions[1:]:
        matched_ref_idx   = {m['idx_a'] for m in other.get('matches_vs_ref', [])}
        matched_other_idx = {m['idx_b'] for m in other.get('matches_vs_ref', [])}

        for m in other.get('matches_vs_ref', []):
            d1 = _px_amps(ref,   m['center_a'])
            d2 = _px_amps(other, m['center_b'])
            pairs = _pairwise_sig([d1, d2])
            if not pairs:
                continue
            y_a = m['peak_a'].get('mean_amp', 0)
            y_b = m['peak_b'].get('mean_amp', 0)
            if y_a < MIN_AMP_DISPLAY and y_b < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='matched', lbl=pairs[0][2],
                x_a=m['center_a'], x_b=m['center_b'],
                y_a=y_a,          y_b=y_b,
                pk_a=m['peak_a'], pk_b=m['peak_b'],
                reg_a=ref,        reg_b=other,
            ))

        for i, pk in enumerate(ref['consensus']):
            if i in matched_ref_idx:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='unmatched_ref', lbl='‡', font_size=18,
                x_a=pk['mean_center'], x_b=None,
                y_a=pk.get('mean_amp', 0), y_b=0,
                pk_a=pk, pk_b=None, reg_a=ref, reg_b=other,
            ))

        for i, pk in enumerate(other['consensus']):
            if i in matched_other_idx:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='unmatched_other', lbl= '‡', font_size=18,
                x_a=None, x_b=pk['mean_center'],
                y_a=0, y_b=pk.get('mean_amp', 0),
                pk_a=None, pk_b=pk, reg_a=ref, reg_b=other,
            ))

    sig_keys = set()
    for sp in sig_pairs:
        if sp['pk_a'] is not None:
            sig_keys.add((id(sp['reg_a']), round(sp['pk_a']['mean_center'], 1)))
        if sp['pk_b'] is not None:
            sig_keys.add((id(sp['reg_b']), round(sp['pk_b']['mean_center'], 1)))

    fig, ax = plt.subplots(figsize=figsize)

    # ── raw / model spectra ───────────────────────────────────────
    if SHOW_RAW_SPECTRA:
        for reg in regions:
            mean_spec = _mean_spectrum(reg)
            label     = f'{reg["name"]} ({"model" if SHOW_MODEL_SPECTRA else "mean"})'
            ax.plot(x_axis, mean_spec,
                    color=reg['color'], lw=1.2, alpha=0.35, label=label)

    # ── fitted peak curves ────────────────────────────────────────
    for reg in regions:
        for pk in reg['consensus']:
            wn  = pk['mean_center']
            key = (id(reg), round(wn, 1))
            is_sig = key in sig_keys
            if not SHOW_ALL_PEAKS and not is_sig:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            curve  = _region_peak_curve(pk)
            color  = palette.get(pk.get('cls'), 'lightgray')
            alpha  = 0.9 if is_sig else 0.3
            lw     = 2.0 if is_sig else 0.8
            ls     = '-'  if is_sig else '--'
            ax.plot(x_axis, curve,
                    color=color, lw=lw, alpha=alpha, ls=ls)
            ax.fill_between(x_axis, curve,
                            color=reg['color'],
                            alpha=0.10 if is_sig else 0.03)

    # ── significance brackets ─────────────────────────────────────
    ax.set_ylim(bottom=0)
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
    tick_h  = y_range * 0.03
    dy_step = y_range * 0.06

    sig_pairs.sort(key=lambda sp: (
        sp['x_a'] if sp['x_a'] is not None else sp['x_b']))

    bracket_tops = {}
    for sp in sig_pairs:
        if sp['kind'] == 'matched':
            x1, x2   = sp['x_a'], sp['x_b']
            y_base   = max(sp['y_a'], sp['y_b'])
            br_color = 'black'
        elif sp['kind'] == 'unmatched_ref':
            x1 = x2  = sp['x_a']
            y_base   = sp['y_a']
            br_color = sp['reg_a']['color']
        else:
            x1 = x2  = sp['x_b']
            y_base   = sp['y_b']
            br_color = sp['reg_b']['color']

        x_mid   = (x1 + x2) / 2
        x_slot  = round(x_mid, 0)
        y_brack = y_base + tick_h * 2
        while bracket_tops.get(x_slot, 0) >= y_brack:
            y_brack += dy_step
        bracket_tops[x_slot] = y_brack + tick_h

        if sp['kind'] == 'matched':
            ax.plot([x1, x1, x2, x2],
                    [y_brack, y_brack + tick_h, y_brack + tick_h, y_brack],
                    lw=0.9, color=br_color, clip_on=False)
        else:
            ax.plot([x1, x1], [y_brack, y_brack + tick_h],
                    lw=0.9, color=br_color, clip_on=False)

        ax.text(x_mid, y_brack + tick_h, sp['lbl'],
                ha='center', va='bottom', fontsize=18,
                fontweight='bold', color=br_color, clip_on=False)

    if bracket_tops:
        ax.set_ylim(0, max(bracket_tops.values()) + dy_step * 2)

    # ── legends ───────────────────────────────────────────────────
    reg_handles = [mpatches.Patch(color=r['color'], label=r['name'])
                   for r in regions]
    cls_handles = [mpatches.Patch(color=palette[k], label=k)
                   for k in PEAK_CLASSES]
    sig_handles = [
        plt.Line2D([0], [0], color='gray', lw=2.0, ls='-',
                   label='sig / unmatched peak'),
        plt.Line2D([0], [0], color='gray', lw=0.8, ls='--',
                   label='non-sig peak'),
    ] if SHOW_ALL_PEAKS else []

    # leg1 = ax.legend(handles=reg_handles + sig_handles,
    #                  loc='upper left',
    #                  fontsize=FONT_SIZES['legend'], title='Region')
    # ax.add_artist(leg1)
    # ax.legend(handles=cls_handles, loc='upper right',
    #           fontsize=FONT_SIZES['legend'], title='Class')

    peak_mode = 'all peaks' if SHOW_ALL_PEAKS else 'sig/unmatched only'
    spec_mode = (f' + {"model" if SHOW_MODEL_SPECTRA else "mean"} spectra'
                 if SHOW_RAW_SPECTRA else '')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Amplitude',          fontsize=FONT_SIZES['axis_label'])
    ax.set_title(f'Spectral Overlay — {peak_mode}{spec_mode}',
                 fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
SHOW_RAW_SPECTRA   = False    # overlay mean raw spectrum for each region
SHOW_ALL_PEAKS     = False   # True = all fitted peaks, False = sig/unmatched only
SHOW_MODEL_SPECTRA = False   # use fitted model cube instead of raw for background

plot_spectra_overlay(regions)

In [ ]:
def plot_mean_spectra(regions, figsize=(12, 5)):
    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sub  = model[r0:r1+1, c0:c1+1, :]
        rH, rW = sub.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)

        pixels    = sub[poly_mask_2d, :]          # (n_px, n_wn)
        mean_spec = pixels.mean(axis=0)
        std_spec  = pixels.std(axis=0)

        ax.plot(x_axis, mean_spec,
                color=reg['color'], lw=2, label=reg['name'])
        ax.fill_between(x_axis,
                        mean_spec - std_spec,
                        mean_spec + std_spec,
                        color=reg['color'], alpha=0.15)

    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Intensity',          fontsize=FONT_SIZES['axis_label'])
    ax.set_title('Mean Model Spectra by Region', fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.legend(fontsize=FONT_SIZES['legend'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_mean_spectra(regions)

In [ ]:
SHOW_RAW_SPECTRA   = True
SHOW_ALL_PEAKS     = False
SHOW_MODEL_SPECTRA = True

def plot_spectra_overlay(regions, figsize=(14, 6)):
    palette    = _cls_palette()
    n          = len(regions)
    jitter_map = {id(reg): jit for reg, jit
                  in zip(regions, np.linspace(-2.5, 2.5, n) if n > 1 else [0.0])}

    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    def _mean_spectrum(reg):
        cube  = model if (SHOW_MODEL_SPECTRA and model is not None) else raw
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)
        sub  = cube[r0:r1+1, c0:c1+1, :]
        rH, rW = sub.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        return sub[poly_mask_2d, :].mean(axis=0)

    def _region_peak_curve(pk, scale):
        """Reconstruct voigt peak scaled to match spectrum amplitude."""
        return scale * np.array(voigt_peak(
            x_axis,
            pk['mean_center'], pk['mean_amp'],
            pk['mean_sigma'],  pk['mean_gamma']))

    # ── collect sig pairs + unmatched ────────────────────────────
    ref       = regions[0]
    sig_pairs = []
    for other in regions[1:]:
        matched_ref_idx   = {m['idx_a'] for m in other.get('matches_vs_ref', [])}
        matched_other_idx = {m['idx_b'] for m in other.get('matches_vs_ref', [])}

        for m in other.get('matches_vs_ref', []):
            d1 = _px_amps(ref,   m['center_a'])
            d2 = _px_amps(other, m['center_b'])
            pairs = _pairwise_sig([d1, d2])
            if not pairs:
                continue
            y_a = m['peak_a'].get('mean_amp', 0)
            y_b = m['peak_b'].get('mean_amp', 0)
            if y_a < MIN_AMP_DISPLAY and y_b < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='matched', lbl=pairs[0][2],
                x_a=m['center_a'], x_b=m['center_b'],
                y_a=y_a,          y_b=y_b,
                pk_a=m['peak_a'], pk_b=m['peak_b'],
                reg_a=ref,        reg_b=other,
            ))

        for i, pk in enumerate(ref['consensus']):
            if i in matched_ref_idx:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='unmatched_ref', lbl=' ★',
                x_a=pk['mean_center'], x_b=None,
                y_a=pk.get('mean_amp', 0), y_b=0,
                pk_a=pk, pk_b=None, reg_a=ref, reg_b=other,
            ))

        for i, pk in enumerate(other['consensus']):
            if i in matched_other_idx:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            sig_pairs.append(dict(
                kind='unmatched_other', lbl= '★',
                x_a=None, x_b=pk['mean_center'],
                y_a=0, y_b=pk.get('mean_amp', 0),
                pk_a=None, pk_b=pk, reg_a=ref, reg_b=other,
            ))

    sig_keys = set()
    for sp in sig_pairs:
        if sp['pk_a'] is not None:
            sig_keys.add((id(sp['reg_a']), round(sp['pk_a']['mean_center'], 1)))
        if sp['pk_b'] is not None:
            sig_keys.add((id(sp['reg_b']), round(sp['pk_b']['mean_center'], 1)))

    fig, ax = plt.subplots(figsize=figsize)

    # ── spectra first — compute scale factor per region ───────────
    # scale maps peak amp units → spectrum units so they overlay cleanly
    region_scales = {}
    spec_max = 0.0
    if SHOW_RAW_SPECTRA:
        for reg in regions:
            mean_spec = _mean_spectrum(reg)
            spec_top  = float(np.nanmax(mean_spec))
            peak_top  = max((pk.get('mean_amp', 0)
                             for pk in reg['consensus']), default=1.0)
            # scale factor: how many spectrum units per peak amp unit
            region_scales[id(reg)] = spec_top / peak_top if peak_top > 0 else 1.0
            spec_max = max(spec_max, spec_top)
            label    = f'{reg["name"]} ({"model" if SHOW_MODEL_SPECTRA else "mean"})'
            ax.plot(x_axis, mean_spec,
                    color=reg['color'], lw=1.5, alpha=0.55,
                    zorder=2, label=label)
            # ax.fill_between(x_axis, mean_spec,
                            # color=reg['color'], alpha=0.08, zorder=1)
    else:
        # no spectra shown — peaks plotted in their own units
        for reg in regions:
            region_scales[id(reg)] = 1.0

    # ── fitted peak curves scaled to spectrum units ───────────────
    for reg in regions:
        scale = region_scales[id(reg)]
        for pk in reg['consensus']:
            wn  = pk['mean_center']
            key = (id(reg), round(wn, 1))
            is_sig = key in sig_keys
            if not SHOW_ALL_PEAKS and not is_sig:
                continue
            if pk.get('mean_amp', 0) < MIN_AMP_DISPLAY:
                continue
            curve = _region_peak_curve(pk, scale)
            color = palette.get(pk.get('cls'), 'lightgray')
            alpha = 0.95 if is_sig else 0.35
            lw    = 2.0  if is_sig else 0.8
            ls    = '-'  if is_sig else '--'
            ax.plot(x_axis, curve,
                    color=color, lw=lw, alpha=alpha, ls=ls, zorder=4)
            ax.fill_between(x_axis, curve,
                            color=reg['color'],
                            alpha=0.12 if is_sig else 0.03, zorder=3)

    # ── ylim from data ────────────────────────────────────────────
    ax.set_ylim(0, spec_max * 1.15 if spec_max > 0 else 0.1)

    # ── significance brackets ─────────────────────────────────────
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
    tick_h  = y_range * 0.03
    dy_step = y_range * 0.06

    sig_pairs.sort(key=lambda sp: (
        sp['x_a'] if sp['x_a'] is not None else sp['x_b']))

    bracket_tops = {}
    for sp in sig_pairs:
        if sp['kind'] == 'matched':
            x1, x2   = sp['x_a'], sp['x_b']
            # scale y_base to spectrum units using ref region scale
            scale    = region_scales[id(sp['reg_a'])]
            y_base   = max(sp['y_a'], sp['y_b']) * scale
            br_color = 'black'
        elif sp['kind'] == 'unmatched_ref':
            x1 = x2  = sp['x_a']
            scale    = region_scales[id(sp['reg_a'])]
            y_base   = sp['y_a'] * scale
            br_color = sp['reg_a']['color']
        else:
            x1 = x2  = sp['x_b']
            scale    = region_scales[id(sp['reg_b'])]
            y_base   = sp['y_b'] * scale
            br_color = sp['reg_b']['color']

        x_mid   = (x1 + x2) / 2
        x_slot  = round(x_mid, 0)
        y_brack = y_base + tick_h * 2
        while bracket_tops.get(x_slot, 0) >= y_brack:
            y_brack += dy_step
        bracket_tops[x_slot] = y_brack + tick_h

        if sp['kind'] == 'matched':
            ax.plot([x1, x1, x2, x2],
                    [y_brack, y_brack + tick_h, y_brack + tick_h, y_brack],
                    lw=0.9, color=br_color, clip_on=False, zorder=5)
        else:
            ax.plot([x1, x1], [y_brack, y_brack + tick_h],
                    lw=0.9, color=br_color, clip_on=False, zorder=5)

        ax.text(x_mid, y_brack + tick_h, sp['lbl'],
                ha='center', va='bottom', fontsize=8,
                fontweight='bold', color=br_color, clip_on=False)

    if bracket_tops:
        ax.set_ylim(0, max(bracket_tops.values()) + dy_step * 2)

    # ── legends ───────────────────────────────────────────────────
    reg_handles = [mpatches.Patch(color=r['color'], label=r['name'])
                   for r in regions]
    cls_handles = [mpatches.Patch(color=palette[k], label=k)
                   for k in PEAK_CLASSES]
    sig_handles = [
        plt.Line2D([0], [0], color='gray', lw=2.0, ls='-',
                   label='sig / unmatched peak'),
        plt.Line2D([0], [0], color='gray', lw=0.8, ls='--',
                   label='non-sig peak'),
    ] if SHOW_ALL_PEAKS else []

    leg1 = ax.legend(handles=reg_handles + sig_handles,
                     loc='upper left',
                     fontsize=FONT_SIZES['legend'], title='Region')
    ax.add_artist(leg1)
    ax.legend(handles=cls_handles, loc='upper right',
              fontsize=FONT_SIZES['legend'], title='Class')

    peak_mode = 'all peaks' if SHOW_ALL_PEAKS else 'sig/unmatched only'
    spec_mode = (f' + {"model" if SHOW_MODEL_SPECTRA else "mean"} spectra'
                 if SHOW_RAW_SPECTRA else '')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Amplitude',          fontsize=FONT_SIZES['axis_label'])
    ax.set_title(f'Spectral Overlay — {peak_mode}{spec_mode}',
                 fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_spectra_overlay(regions)

In [ ]:
def plot_spatial_sig_peaks(regions, mode='amplitude', figsize_per_map=(3, 3)):
    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    def _prev_vals(reg, center_wn):
        sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return m.any(axis=2)[poly_mask_2d].astype(float)

    def _make_amp_map(reg, ref_wn, mode):
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)
        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        poly_mask    = poly_mask_2d[:, :, np.newaxis]
        mask = (sa > AMP_THRESHOLD) & \
               (np.abs(sc - ref_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        if mode == 'amplitude':
            amp_map = (sa * mask).max(axis=2).astype(float)
        else:
            amp_map = mask.any(axis=2).astype(float)
        amp_map[~poly_mask_2d] = np.nan
        return amp_map

    ref = regions[0]
    rows = []   # each entry: dict with all info needed to plot one row

    for other in regions[1:]:
        matched_ref_idx   = {m['idx_a'] for m in other.get('matches_vs_ref', [])}
        matched_other_idx = {m['idx_b'] for m in other.get('matches_vs_ref', [])}

        # ── matched pairs — test for significant difference ───────────────
        for m in other.get('matches_vs_ref', []):
            if mode == 'amplitude':
                d1 = _px_amps(ref,   m['center_a'])
                d2 = _px_amps(other, m['center_b'])
            else:
                d1 = _prev_vals(ref,   m['center_a'])
                d2 = _prev_vals(other, m['center_b'])

            pairs = _pairwise_sig([d1, d2])
            if not pairs:
                continue
            rows.append(dict(
                kind     = 'matched',
                lbl      = pairs[0][2],
                ref_wn   = (m['center_a'] + m['center_b']) / 2,
                center_a = m['center_a'],
                center_b = m['center_b'],
                reg_b    = other,
                note     = f'{m["center_a"]:.0f} cm⁻¹ (matched) {pairs[0][2]}',
            ))

        # ── unmatched peaks in reference — absent in this other region ────
        for i, pk in enumerate(ref['consensus']):
            if i in matched_ref_idx:
                continue
            if pk.get(mode.replace('amplitude', 'mean_amp')
                         .replace('prevalence', 'prevalence'), 0) < MIN_AMP_DISPLAY:
                continue
            rows.append(dict(
                kind     = 'unmatched_ref',
                lbl      = 'absent',
                ref_wn   = pk['mean_center'],
                center_a = pk['mean_center'],
                center_b = None,
                reg_b    = other,
                note     = f'{pk["mean_center"]:.0f} cm⁻¹  only in {ref["name"]}',
            ))

        # ── unmatched peaks in other — absent in reference ────────────────
        for i, pk in enumerate(other['consensus']):
            if i in matched_other_idx:
                continue
            if pk.get(mode.replace('amplitude', 'mean_amp')
                         .replace('prevalence', 'prevalence'), 0) < MIN_AMP_DISPLAY:
                continue
            rows.append(dict(
                kind     = 'unmatched_other',
                lbl      = 'absent',
                ref_wn   = pk['mean_center'],
                center_a = None,
                center_b = pk['mean_center'],
                reg_b    = other,
                note     = f'{pk["mean_center"]:.0f} cm⁻¹  only in {other["name"]}',
            ))

    if not rows:
        print(f'Nothing to plot for mode={mode}.')
        return

    n_reg   = len(regions)
    n_rows  = len(rows)
    fig, axes = plt.subplots(
        n_rows, n_reg,
        figsize=(figsize_per_map[0] * n_reg,
                 figsize_per_map[1] * n_rows),
        squeeze=False)

    for row_i, sp in enumerate(rows):
        ref_wn = sp['ref_wn']
        maps   = []
        for reg in regions:
            # for unmatched peaks absent in this region, show blank nan map
            if sp['kind'] == 'unmatched_other' and reg is ref:
                # ref doesn't have this peak — show blank
                sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
                blank = np.full(sa.shape[:2], np.nan)
                maps.append(blank)
            elif sp['kind'] == 'unmatched_ref' and reg is sp['reg_b']:
                # other doesn't have this peak — show blank
                sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
                blank = np.full(sa.shape[:2], np.nan)
                maps.append(blank)
            else:
                maps.append(_make_amp_map(reg, ref_wn, mode))

        vmin = np.nanmin([np.nanmin(m) for m in maps if not np.all(np.isnan(m))])
        vmax = np.nanmax([np.nanmax(m) for m in maps if not np.all(np.isnan(m))])

        for col, (reg, amp_map) in enumerate(zip(regions, maps)):
            ax  = axes[row_i][col]
            im  = ax.imshow(amp_map, cmap=plasma_rp,
                            vmin=vmin, vmax=vmax, aspect='auto')
            plt.colorbar(im, ax=ax, shrink=0.8,
                         label='Amplitude' if mode == 'amplitude' else 'Present')

            # title: color red if this region is the absent one
            is_absent = (sp['kind'] == 'unmatched_other' and reg is ref) or \
                        (sp['kind'] == 'unmatched_ref'   and reg is sp['reg_b'])
            title_sfx = '  [absent]' if is_absent else ''
            ax.set_title(reg['name'] + title_sfx,
                         color='red' if is_absent else reg['color'],
                         fontweight='bold', fontsize=9)
            ax.set_xlabel('Col', fontsize=8)
            ax.tick_params(labelsize=7)

        axes[row_i][0].set_ylabel(f'{sp["note"]}\nRow', fontsize=8, fontweight='bold')

    mode_lbl = 'Amplitude' if mode == 'amplitude' else 'Prevalence'
    fig.suptitle(
        f'Spatial Heatmaps — Significant & Unmatched Peaks ({mode_lbl})',
        fontsize=12)
    plt.tight_layout()
    plt.show()

    n_matched   = sum(1 for r in rows if r['kind'] == 'matched')
    n_unmatched = sum(1 for r in rows if r['kind'] != 'matched')
    print(f'{n_matched} significantly different matched pairs, '
          f'{n_unmatched} unmatched (absent) peaks.')


plot_spatial_sig_peaks(regions, mode='amplitude')
plot_spatial_sig_peaks(regions, mode='prevalence')

In [ ]:
def plot_diff_lollipop(reg_a, reg_b, figsize=(12, 5)):
    matches = match_peaks(reg_a, reg_b)
    if not matches:
        print('No matched peaks — try increasing CENTER_MATCH_THRESHOLD.')
        return
    fig, ax = plt.subplots(figsize=figsize)
    palette = _cls_palette()

    dy_vals = []
    for m in matches:
        x      = (m['center_a'] + m['center_b']) / 2
        dy     = m['peak_b']['mean_amp'] - m['peak_a']['mean_amp']
        stem_c = reg_b['color'] if dy >= 0 else reg_a['color']
        cls    = m['peak_a'].get('cls') or m['peak_b'].get('cls')
        head_c = palette.get(cls, 'lightgray')
        ax.vlines(x, 0, dy, color=stem_c, lw=2, alpha=0.8)
        ax.scatter(x, dy, s=60, color=head_c, zorder=5,
                   edgecolors=stem_c, linewidths=0.8)
        dy_vals.append(abs(dy))

    nonzero = [v for v in dy_vals if v > 0]
    linthresh = float(np.percentile(nonzero, 10)) if nonzero else 1e-4
    ax.set_yscale('symlog', linthresh=linthresh, linscale=0.5)
    ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    # ax.legend(handles=cls_handles, loc='upper right',
    #           fontsize=FONT_SIZES['legend'], title='Class')

    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    name_a, name_b = reg_a['name'], reg_b['name']
    ax.set_ylabel(f'Δ Amplitude  ({name_b} − {name_a})  [symlog]',
                  fontsize=FONT_SIZES['axis_label'])
    ax.set_title(f'Differential Lollipop: {name_a}  vs  {name_b}',
                 fontsize=FONT_SIZES['subtitle'])
    ax.text(0.01, 1.02, f'▲ stronger in {name_b}',
            transform=ax.transAxes, color=reg_b['color'],
            fontsize=FONT_SIZES['annotation'], va='top')
    ax.text(0.01, 0.00, f'▼ stronger in {name_a}',
            transform=ax.transAxes, color=reg_a['color'],
            fontsize=FONT_SIZES['annotation'], va='bottom')
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_diff_lollipop(regions[0], regions[1])

In [ ]:
def plot_violin_by_class(regions, class_def=PEAK_CLASSES, figsize=None):
    cls_list = list(class_def.keys())
    if figsize is None:
        figsize = (max(4 * len(cls_list), 8), 5)
    fig, axes = plt.subplots(1, len(cls_list), figsize=figsize, sharey=False)
    if len(cls_list) == 1:
        axes = [axes]

    for ax, cls in zip(axes, cls_list):
        data, labels = [], []
        for reg in regions:
            sa, sc, poly_mask, _ = _poly_slices(reg)
            mask = _class_wn_mask(sa, sc, class_def[cls]) & poly_mask
            vals = sa[mask]
            data.append(vals if len(vals) > 1 else np.array([0.0, 0.0]))
            labels.append(reg['name'])

        parts = ax.violinplot(data, showmedians=True, showextrema=True)
        for pc, reg in zip(parts['bodies'], regions):
            pc.set_facecolor(reg['color'])
            pc.set_alpha(0.6)
        for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
            if key in parts:
                parts[key].set_color('black')
                parts[key].set_linewidth(0.8)
        ax.set_xticks(range(1, len(regions) + 1))
        ax.set_xticklabels(labels, rotation=20, ha='right',
                           fontsize=FONT_SIZES['tick_label'])
        ax.tick_params(axis='y', labelsize=FONT_SIZES['tick_label'])
        ax.set_title(cls, fontsize=FONT_SIZES['subtitle'])
        ax.spines[['top', 'right']].set_visible(False)

        pairs = _pairwise_sig(data)
        if pairs:
            x_positions = list(range(1, len(regions) + 1))
            nonempty = [d for d in data if len(d) > 1]
            y_top = max(d.max() for d in nonempty) * 1.05 if nonempty else 0.01
            dy    = max(d.max() for d in nonempty) * 0.12 if nonempty else 0.01
            _draw_sig_brackets(ax, pairs, x_positions, y_top, dy=dy,
                               fontsize=FONT_SIZES['annotation'])

    axes[0].set_ylabel('Peak Amplitude', fontsize=FONT_SIZES['axis_label'])
    fig.suptitle('Amplitude Distribution by Peak Class and Region',
                 fontsize=FONT_SIZES['title'])
    plt.tight_layout()
    plt.show()

plot_violin_by_class(regions)


In [ ]:
def plot_boxplot_matched(regions, figsize=(13, 5)):
    if len(regions) < 2:
        print('Need at least 2 regions.')
        return
    ref     = regions[0]
    ref_pks = ref['consensus']
    if not ref_pks:
        print('Reference region has no consensus peaks.')
        return

    n_pk  = len(ref_pks)
    n_reg = len(regions)
    width = 0.8 / n_reg
    offs  = np.linspace(-0.4 + width / 2, 0.4 - width / 2, n_reg)
    x_pos = np.arange(n_pk)
    fig, ax = plt.subplots(figsize=figsize)

    def _pixel_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.array([0.0])

    all_per_peak = []
    for reg in regions:
        per_peak = []
        if reg is ref:
            for pk in ref_pks:
                per_peak.append(_pixel_amps(reg, pk['mean_center']))
        else:
            mm = {m['idx_a']: m for m in reg.get('matches_vs_ref', [])}
            for i, pk in enumerate(ref_pks):
                ctr = mm[i]['center_b'] if i in mm else None
                per_peak.append(_pixel_amps(reg, ctr) if ctr is not None else np.array([0.0]))
        all_per_peak.append(per_peak)

    for reg, off, per_peak in zip(regions, offs, all_per_peak):
        bp = ax.boxplot(
            per_peak, positions=x_pos + off, widths=width * 0.85,
            patch_artist=True, manage_ticks=False,
            medianprops=dict(color='black', lw=1.5),
            whiskerprops=dict(color=reg['color']),
            capprops=dict(color=reg['color']),
            flierprops=dict(marker='.', markerfacecolor=reg['color'],
                            markersize=3, alpha=0.4))
        for patch in bp['boxes']:
            patch.set_facecolor(reg['color'])
            patch.set_alpha(0.55)

    real_vals  = [d for pp in all_per_peak for d in pp if len(d) > 1]
    global_max = max(d.max() for d in real_vals) if real_vals else 0.1
    dy_global  = global_max * 0.08

    for i_pk in range(n_pk):
        peak_data      = [all_per_peak[r][i_pk] for r in range(n_reg)]
        peak_data_test = [d if len(d) > 1 else np.zeros(5) for d in peak_data]
        pairs = _pairwise_sig(peak_data_test, threshold=0.05)
        if pairs:
            x_positions = [float(x_pos[i_pk]) + offs[r] for r in range(n_reg)]
            real_here = [d for d in peak_data if len(d) > 1]
            y_top = (max(np.percentile(d, 95) for d in real_here) * 1.05
                     if real_here else dy_global * 0.5)
            _draw_sig_brackets(ax, pairs, x_positions, y_top, dy=dy_global,
                               fontsize=FONT_SIZES['annotation'])

    ax.set_xticks(x_pos)
    tick_labels = [f'{p["mean_center"]:.0f} cm⁻\xb9' for p in ref_pks]
    ax.set_xticklabels(tick_labels, rotation=45, ha='right',
                       fontsize=FONT_SIZES['tick_label'])
    ax.tick_params(axis='y', labelsize=FONT_SIZES['tick_label'])
    ax.set_ylabel('Peak Amplitude', fontsize=FONT_SIZES['axis_label'])
    ax.set_title('Matched Peak Amplitude Distribution per Region',
                 fontsize=FONT_SIZES['subtitle'])
    handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    ax.legend(handles=handles, fontsize=FONT_SIZES['legend'])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_boxplot_matched(regions)

In [ ]:
def plot_spatial_heatmaps(
        regions,
        target='class',
        class_name=None,
        peak_center=None,
        cmap=plasma_rp,
        figsize=None):
    n = len(regions)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]

    maps, vmin, vmax = [], np.inf, -np.inf
    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        poly_mask    = poly_mask_2d[:, :, np.newaxis]

        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name]) & poly_mask
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > AMP_THRESHOLD) & \
                      (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD) & poly_mask
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = (sa * poly_mask).max(axis=2)

        amp_map[~poly_mask_2d] = np.nan

        maps.append(amp_map)
        vmin = min(vmin, np.nanmin(amp_map))
        vmax = max(vmax, np.nanmax(amp_map))

    for ax, reg, amp_map in zip(axes, regions, maps):
        im = ax.imshow(amp_map, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_title(reg['name'], color=reg['color'], fontweight='bold',
                     fontsize=FONT_SIZES['subtitle'])
        ax.set_xlabel('Col', fontsize=FONT_SIZES['axis_label'])
        ax.set_ylabel('Row', fontsize=FONT_SIZES['axis_label'])
        ax.tick_params(labelsize=FONT_SIZES['tick_label'])
        plt.colorbar(im, ax=ax, shrink=0.8, label='Amplitude')

    if class_name:
        title = f'Spatial Heatmap — {class_name}'
    elif peak_center is not None:
        title = f'Spatial Heatmap — {peak_center:.0f} cm⁻¹'
    else:
        title = 'Spatial Heatmap — Max Peak Amplitude'
    fig.suptitle(title, fontsize=FONT_SIZES['title'])
    plt.tight_layout()
    plt.show()

for cls_name in PEAK_CLASSES:
    plot_spatial_heatmaps(regions, target='class', class_name=cls_name)

In [ ]:
for pk in regions[0]['consensus']:
    if pk.get('cls') == 'nucleotides':
        plot_spatial_heatmaps(regions, target='peak', peak_center=pk['mean_center'])

In [ ]:
from scipy.ndimage import distance_transform_edt

def plot_intensity_vs_boundary_distance(
        regions,
        target='class',          # 'class' or 'peak'
        class_name=None,
        peak_center=None,
        n_bins=20,
        figsize=(10, 5)):

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)

        # distance transform — each pixel gets its distance to nearest boundary
        dist_map = distance_transform_edt(poly_mask_2d)   # (rH, rW)

        # amplitude map for the target
        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name])
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > AMP_THRESHOLD) & \
                      (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD)
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = (sa * poly_mask_2d[:, :, np.newaxis]).max(axis=2)

        # bin by distance
        max_dist  = dist_map[poly_mask_2d].max()
        bin_edges = np.linspace(0, max_dist, n_bins + 1)
        bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

        means = []
        sems  = []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = poly_mask_2d & (dist_map >= lo) & (dist_map < hi)
            vals   = amp_map[in_bin]
            means.append(float(vals.mean()) if len(vals) > 0 else np.nan)
            sems.append(float(vals.std() / np.sqrt(len(vals)))
                        if len(vals) > 1 else np.nan)

        means = np.array(means)
        sems  = np.array(sems)

        ax.plot(bin_ctrs, means,
                color=reg['color'], lw=2, label=reg['name'])
        ax.fill_between(bin_ctrs,
                        means - sems, means + sems,
                        color=reg['color'], alpha=0.2)

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Mean Amplitude',                  fontsize=FONT_SIZES['axis_label'])
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=FONT_SIZES['legend'])
    ax.semilogy()
    if target == 'class' and class_name:
        ax.set_title(f'Intensity vs Boundary Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Intensity vs Boundary Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Intensity vs Boundary Distance — Max Amplitude',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()


# per class
for cls_name in PEAK_CLASSES:
    plot_intensity_vs_boundary_distance(regions, target='class', class_name=cls_name)

# or a specific peak
# plot_intensity_vs_boundary_distance(regions, target='peak', peak_center=1004.0)

In [ ]:
plot_intensity_vs_boundary_distance(regions, target='peak', peak_center=907)

In [ ]:
from scipy.ndimage import distance_transform_edt
from scipy.stats import linregress
from statsmodels.nonparametric.smoothers_lowess import lowess

def plot_intensity_vs_boundary_distance(
        regions,
        best_fit ='lowess',
        target='class',
        class_name=None,
        peak_center=None,
        n_bins=20,
        figsize=(10, 5)):

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        dist_map     = distance_transform_edt(poly_mask_2d)

        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name])
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > AMP_THRESHOLD) & \
                      (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD)
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = (sa * poly_mask_2d[:, :, np.newaxis]).max(axis=2)

        # bin pixels by distance → one point per bin
        max_dist  = dist_map[poly_mask_2d].max()
        bin_edges = np.linspace(0, max_dist, n_bins + 1)
        bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

        means, sems = [], []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = poly_mask_2d & (dist_map >= lo) & (dist_map < hi)
            vals   = amp_map[in_bin]
            means.append(float(vals.mean()) if len(vals) > 0 else np.nan)
            sems.append(float(vals.std() / np.sqrt(len(vals)))
                        if len(vals) > 1 else np.nan)

        means    = np.array(means)
        sems     = np.array(sems)
        valid    = ~np.isnan(means)
        x_valid  = bin_ctrs[valid]
        y_valid  = means[valid]

        # scatter — one point per distance bin
        ax.scatter(x_valid, y_valid,
                   color=reg['color'], s=40, alpha=0.8,
                   edgecolors='none', zorder=3,
                   label=reg['name'])

        # error bars
        ax.errorbar(x_valid, y_valid, yerr=sems[valid],
                    fmt='none', ecolor=reg['color'],
                    elinewidth=0.8, capsize=2, alpha=0.5, zorder=2)

        #  best fit
        if len(x_valid) >= 2:
            if best_fit == "linear":
                slope, intercept, r, p, _ = linregress(x_valid, y_valid)
                x_fit = np.linspace(x_valid.min(), x_valid.max(), 200)
                y_fit = slope * x_fit + intercept
                ax.plot(x_fit, y_fit,
                color=reg['color'], lw=2, ls='--', zorder=4,
                label=f'{reg["name"]} fit  r²={r**2:.2f}  p={p:.3f}')
            if best_fit == 'lowess':
                smoothed = lowess(y_valid, x_valid, frac=0.4)
                x_fit, y_fit = smoothed[:, 0], smoothed[:, 1]
                ax.plot(x_fit, y_fit,
                color=reg['color'], lw=2, ls='--', zorder=4,
                label=f'{reg["name"]} fit')
            

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Mean Amplitude',                  fontsize=FONT_SIZES['axis_label'])
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=FONT_SIZES['legend'])

    if target == 'class' and class_name:
        ax.set_title(f'Intensity vs Boundary Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Intensity vs Boundary Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Intensity vs Boundary Distance — Max Amplitude',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()

for cls_name in PEAK_CLASSES:
    plot_intensity_vs_boundary_distance(regions, target='class', class_name=cls_name)

In [ ]:
REGRESSION   = 'lowess'    # 'linear', 'poly', 'exp', 'log', 'lowess'
POLY_DEGREE  = 2           # only used when REGRESSION == 'poly'
LOWESS_FRAC  = 0.4         # only used when REGRESSION == 'lowess'

def plot_intensity_vs_boundary_distance(
        regions,
        target='class',
        class_name=None,
        peak_center=None,
        n_bins=100,
        figsize=(8, 5)):
    from scipy.stats   import linregress
    from scipy.ndimage import distance_transform_edt

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        dist_map     = distance_transform_edt(poly_mask_2d)

        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name])
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > AMP_THRESHOLD) & \
                      (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD)
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = (sa * poly_mask_2d[:, :, np.newaxis]).max(axis=2)

        # bin by distance — normalize to per-pixel mean for comparability
        max_dist  = dist_map[poly_mask_2d].max()
        bin_edges = np.linspace(0, max_dist, n_bins + 1)
        bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

        means, sems = [], []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = poly_mask_2d & (dist_map >= lo) & (dist_map < hi)
            vals   = amp_map[in_bin]                  # already per-pixel
            means.append(float(vals.mean()) if len(vals) > 0 else np.nan)
            sems.append(float(vals.std() / np.sqrt(len(vals)))
                        if len(vals) > 1 else np.nan)

        means   = np.array(means)
        sems    = np.array(sems)
        valid   = ~np.isnan(means)
        x_valid = bin_ctrs[valid]
        y_valid = means[valid]

        ax.scatter(x_valid, y_valid,
                   color=reg['color'], s=40, alpha=0.8,
                   edgecolors='none', zorder=3, label=reg['name'])
        ax.errorbar(x_valid, y_valid, yerr=sems[valid],
                    fmt='none', ecolor=reg['color'],
                    elinewidth=0.8, capsize=2, alpha=0.5, zorder=2)

        # ── regression ───────────────────────────────────────────
        x_fit   = np.linspace(x_valid.min(), x_valid.max(), 200)
        fit_lbl = reg['name']
        y_fit   = None

        if REGRESSION == 'linear' and len(x_valid) >= 2:
            slope, intercept, r, p, _ = linregress(x_valid, y_valid)
            y_fit   = slope * x_fit + intercept
            fit_lbl += f'  r²={r**2:.2f}  p={p:.3f}'

        elif REGRESSION == 'poly' and len(x_valid) >= POLY_DEGREE + 1:
            coeffs  = np.polyfit(x_valid, y_valid, deg=POLY_DEGREE)
            y_fit   = np.polyval(coeffs, x_fit)
            y_pred  = np.polyval(coeffs, x_valid)
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (deg {POLY_DEGREE})'

        elif REGRESSION == 'exp' and len(x_valid) >= 3:
            from scipy.optimize import curve_fit
            def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
            try:
                popt, _ = curve_fit(exp_decay, x_valid, y_valid,
                                    p0=[y_valid.max(), 0.1, y_valid.min()],
                                    maxfev=5000)
                y_fit   = exp_decay(x_fit, *popt)
                y_pred  = exp_decay(x_valid, *popt)
                ss_res  = np.sum((y_valid - y_pred)**2)
                ss_tot  = np.sum((y_valid - y_valid.mean())**2)
                r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                fit_lbl += f'  r²={r2:.2f}  (exp)'
            except RuntimeError:
                print(f'Exp fit failed for {reg["name"]}')

        elif REGRESSION == 'log' and len(x_valid) >= 2:
            coeffs  = np.polyfit(np.log(x_valid + 1), y_valid, deg=1)
            y_fit   = coeffs[0] * np.log(x_fit + 1) + coeffs[1]
            y_pred  = coeffs[0] * np.log(x_valid + 1) + coeffs[1]
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (log)'

        elif REGRESSION == 'lowess' and len(x_valid) >= 3:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            smoothed       = lowess(y_valid, x_valid, frac=LOWESS_FRAC)
            x_fit, y_fit   = smoothed[:, 0], smoothed[:, 1]
            fit_lbl       += '  (lowess)'

        if y_fit is not None:
            ax.plot(x_fit, y_fit,
                    color=reg['color'], lw=2, ls='--',
                    zorder=4, label=fit_lbl)

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Mean Amplitude per Pixel',        fontsize=FONT_SIZES['axis_label'])
    ax.set_ylim(0.025,0.13)
    ax.spines[['top', 'right']].set_visible(False)
    # ax.legend(fontsize=FONT_SIZES['legend'])

    if target == 'class' and class_name:
        ax.set_title(f'Intensity vs Boundary Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Intensity vs Boundary Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Intensity vs Boundary Distance — Max Amplitude',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()

# for cls_name in PEAK_CLASSES:
#     plot_intensity_vs_boundary_distance(regions, target='class', class_name=cls_name)

In [ ]:
REGRESSION   = 'lowess'    # 'linear', 'poly', 'exp', 'log', 'lowess'

def plot_intensity_vs_boundary_distance(
        regions,
        target='class',
        class_name=None,
        peak_center=None,
        figsize=(10, 5)):
    from scipy.stats   import linregress
    from scipy.ndimage import distance_transform_edt

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        dist_map     = distance_transform_edt(poly_mask_2d)

        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name])
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > AMP_THRESHOLD) & \
                      (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD)
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = (sa * poly_mask_2d[:, :, np.newaxis]).max(axis=2)

        # extract per-pixel values inside polygon
        x_all = dist_map[poly_mask_2d]    # distance for every pixel
        y_all = amp_map[poly_mask_2d]     # amplitude for every pixel

        # scatter every pixel — alpha low since there will be many overlapping
        ax.scatter(x_all, y_all,
                   color=reg['color'], s=6, alpha=0.15,
                   edgecolors='none', zorder=2, label=reg['name'])

        # ── regression over raw pixel data ───────────────────────
        valid   = ~np.isnan(y_all)
        x_valid = x_all[valid]
        y_valid = y_all[valid]
        x_fit   = np.linspace(x_valid.min(), x_valid.max(), 200)
        fit_lbl = reg['name']
        y_fit   = None

        if REGRESSION == 'linear' and len(x_valid) >= 2:
            slope, intercept, r, p, _ = linregress(x_valid, y_valid)
            y_fit   = slope * x_fit + intercept
            fit_lbl += f'  r²={r**2:.2f}  p={p:.3f}'

        elif REGRESSION == 'poly' and len(x_valid) >= POLY_DEGREE + 1:
            coeffs  = np.polyfit(x_valid, y_valid, deg=POLY_DEGREE)
            y_fit   = np.polyval(coeffs, x_fit)
            y_pred  = np.polyval(coeffs, x_valid)
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (deg {POLY_DEGREE})'

        elif REGRESSION == 'exp' and len(x_valid) >= 3:
            from scipy.optimize import curve_fit
            def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
            try:
                popt, _ = curve_fit(exp_decay, x_valid, y_valid,
                                    p0=[y_valid.max(), 0.1, y_valid.min()],
                                    maxfev=5000)
                y_fit   = exp_decay(x_fit, *popt)
                y_pred  = exp_decay(x_valid, *popt)
                ss_res  = np.sum((y_valid - y_pred)**2)
                ss_tot  = np.sum((y_valid - y_valid.mean())**2)
                r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                fit_lbl += f'  r²={r2:.2f}  (exp)'
            except RuntimeError:
                print(f'Exp fit failed for {reg["name"]}')

        elif REGRESSION == 'log' and len(x_valid) >= 2:
            coeffs  = np.polyfit(np.log(x_valid + 1), y_valid, deg=1)
            y_fit   = coeffs[0] * np.log(x_fit + 1) + coeffs[1]
            y_pred  = coeffs[0] * np.log(x_valid + 1) + coeffs[1]
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (log)'

        elif REGRESSION == 'lowess' and len(x_valid) >= 3:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            smoothed     = lowess(y_valid, x_valid, frac=LOWESS_FRAC)
            x_fit, y_fit = smoothed[:, 0], smoothed[:, 1]
            fit_lbl     += '  (lowess)'

        if y_fit is not None:
            ax.plot(x_fit, y_fit,
                    color=reg['color'], lw=2.5, ls='--',
                    zorder=4, label=fit_lbl)

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Amplitude per Pixel',             fontsize=FONT_SIZES['axis_label'])
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=FONT_SIZES['legend'])

    if target == 'class' and class_name:
        ax.set_title(f'Intensity vs Boundary Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Intensity vs Boundary Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Intensity vs Boundary Distance — Max Amplitude',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()

for cls_name in PEAK_CLASSES:
    plot_intensity_vs_boundary_distance(regions, target='class', class_name=cls_name)

In [ ]:
REGRESSION   = 'lowess'    # 'linear', 'poly', 'exp', 'log', 'lowess'
POLY_DEGREE  = 2           # only used when REGRESSION == 'poly'
LOWESS_FRAC  = 0.4         # only used when REGRESSION == 'lowess'

def plot_intensity_vs_boundary_distance(
        regions,
        target='class',
        class_name=None,
        peak_center=None,
        n_bins=50,
        figsize=(8, 4),
        y_margin=0.1,
        match_threshold=CENTER_MATCH_THRESHOLD,
        min_amp=AMP_THRESHOLD):
    from scipy.stats   import linregress
    from scipy.ndimage import distance_transform_edt

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        dist_map     = distance_transform_edt(poly_mask_2d)

        if target == 'class' and class_name in PEAK_CLASSES:
            mask    = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name])
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask    = (sa > min_amp) & \
                      (np.abs(sc - peak_center) < match_threshold)
            mask   &= poly_mask_2d[:, :, np.newaxis]
            amp_map = (sa * mask).max(axis=2)

        else:
            amp_map = (sa * poly_mask_2d[:, :, np.newaxis]).max(axis=2)

        # bin by distance — normalize to per-pixel mean for comparability
        max_dist  = dist_map[poly_mask_2d].max()
        bin_edges = np.linspace(0, max_dist, n_bins + 1)
        bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

        means, sems = [], []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = poly_mask_2d & (dist_map >= lo) & (dist_map < hi)
            vals   = amp_map[in_bin]                  # already per-pixel
            means.append(float(vals.mean()) if len(vals) > 0 else np.nan)
            sems.append(float(vals.std() / np.sqrt(len(vals)))
                        if len(vals) > 1 else np.nan)

        means   = np.array(means)
        sems    = np.array(sems)
        valid   = ~np.isnan(means)
        x_valid = bin_ctrs[valid]
        y_valid = means[valid]

        ax.scatter(x_valid, y_valid,
                   color=reg['color'], s=40, alpha=0.8,
                   edgecolors='none', zorder=3, label=reg['name'])
        ax.errorbar(x_valid, y_valid, yerr=sems[valid],
                    fmt='none', ecolor=reg['color'],
                    elinewidth=0.8, capsize=2, alpha=0.5, zorder=2)

        # ── regression ───────────────────────────────────────────
        x_fit   = np.linspace(x_valid.min(), x_valid.max(), 200)
        fit_lbl = reg['name']
        y_fit   = None

        if REGRESSION == 'linear' and len(x_valid) >= 2:
            slope, intercept, r, p, _ = linregress(x_valid, y_valid)
            y_fit   = slope * x_fit + intercept
            fit_lbl += f'  r²={r**2:.2f}  p={p:.3f}'

        elif REGRESSION == 'poly' and len(x_valid) >= POLY_DEGREE + 1:
            coeffs  = np.polyfit(x_valid, y_valid, deg=POLY_DEGREE)
            y_fit   = np.polyval(coeffs, x_fit)
            y_pred  = np.polyval(coeffs, x_valid)
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (deg {POLY_DEGREE})'

        elif REGRESSION == 'exp' and len(x_valid) >= 3:
            from scipy.optimize import curve_fit
            def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
            try:
                popt, _ = curve_fit(exp_decay, x_valid, y_valid,
                                    p0=[y_valid.max(), 0.1, y_valid.min()],
                                    maxfev=5000)
                y_fit   = exp_decay(x_fit, *popt)
                y_pred  = exp_decay(x_valid, *popt)
                ss_res  = np.sum((y_valid - y_pred)**2)
                ss_tot  = np.sum((y_valid - y_valid.mean())**2)
                r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                fit_lbl += f'  r²={r2:.2f}  (exp)'
            except RuntimeError:
                print(f'Exp fit failed for {reg["name"]}')

        elif REGRESSION == 'log' and len(x_valid) >= 2:
            coeffs  = np.polyfit(np.log(x_valid + 1), y_valid, deg=1)
            y_fit   = coeffs[0] * np.log(x_fit + 1) + coeffs[1]
            y_pred  = coeffs[0] * np.log(x_valid + 1) + coeffs[1]
            ss_res  = np.sum((y_valid - y_pred)**2)
            ss_tot  = np.sum((y_valid - y_valid.mean())**2)
            r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            fit_lbl += f'  r²={r2:.2f}  (log)'

        elif REGRESSION == 'lowess' and len(x_valid) >= 3:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            smoothed       = lowess(y_valid, x_valid, frac=LOWESS_FRAC)
            x_fit, y_fit   = smoothed[:, 0], smoothed[:, 1]
            fit_lbl       += '  (lowess)'

        if y_fit is not None:
            ax.plot(x_fit, y_fit,
                    color=reg['color'], lw=2, ls='--',
                    zorder=4, label=fit_lbl)

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Mean Amplitude per Pixel',        fontsize=FONT_SIZES['axis_label'])
    # ax.set_ylim(0.025,0.13)
    ax.spines[['top', 'right']].set_visible(False)
    # ax.legend(fontsize=FONT_SIZES['legend'])

    if target == 'class' and class_name:
        ax.set_title(f'Intensity vs Boundary Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Intensity vs Boundary Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Intensity vs Boundary Distance — Max Amplitude',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()

# for cls_name in PEAK_CLASSES:
#     plot_intensity_vs_boundary_distance(regions, target='class', class_name=cls_name)

In [ ]:
plot_intensity_vs_boundary_distance(regions, target='peak', peak_center=907)

In [ ]:
TARGET_PEAKS = [906.85, 1442.56, 1462.37]

for wn in TARGET_PEAKS:
    plot_intensity_vs_boundary_distance(
        regions, target='peak', peak_center=wn,
        match_threshold=25.0,    # override CENTER_MATCH_THRESHOLD locally
        min_amp=0.005)           # override MIN_AMP_DISPLAY locally

In [ ]:
def plot_intensity_vs_boundary_distance(
        regions,
        target='class',
        class_name=None,
        peak_center=None,
        n_bins=50,
        figsize=(18, 4),
        y_margin=0.1,
        match_threshold=CENTER_MATCH_THRESHOLD,
        min_amp=AMP_THRESHOLD):
    from scipy.stats   import linregress
    from scipy.ndimage import distance_transform_edt

    fig, ax = plt.subplots(figsize=figsize)

    for reg in regions:
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)

        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]

        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        poly_mask    = poly_mask_2d[:, :, np.newaxis]
        dist_map     = distance_transform_edt(poly_mask_2d)

        if target == 'class' and class_name in PEAK_CLASSES:
            peak_mask = _class_wn_mask(sa, sc, PEAK_CLASSES[class_name]) & poly_mask
            amp_map   = (sa * peak_mask).sum(axis=2)
            prev_map  = peak_mask.any(axis=2)
        elif target == 'peak' and peak_center is not None:
            peak_mask = (sa > min_amp) & \
                        (np.abs(sc - peak_center) < match_threshold) & poly_mask
            amp_map   = (sa * peak_mask).max(axis=2)
            prev_map  = peak_mask.any(axis=2)
        else:
            amp_map  = (sa * poly_mask).max(axis=2)
            prev_map = (sa * poly_mask > min_amp).any(axis=2)

        max_dist  = dist_map[poly_mask_2d].max()
        bin_edges = np.linspace(0, max_dist, n_bins + 1)
        bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

        weighted, sems = [], []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = poly_mask_2d & (dist_map >= lo) & (dist_map < hi)
            if not in_bin.any():
                weighted.append(np.nan)
                sems.append(np.nan)
                continue
            amp_vals  = amp_map[in_bin]
            prev_vals = prev_map[in_bin].astype(float)
            w = float(amp_vals.mean() * prev_vals.mean())
            weighted.append(w)
            # SEM propagated through product: σ_w ≈ prev*σ_amp + amp*σ_prev
            sem_amp  = amp_vals.std()  / np.sqrt(len(amp_vals))
            sem_prev = prev_vals.std() / np.sqrt(len(prev_vals)) \
                       if len(prev_vals) > 1 else 0.0
            sems.append(float(prev_vals.mean() * sem_amp +
                               amp_vals.mean()  * sem_prev))

        weighted = np.array(weighted)
        sems     = np.array(sems)
        valid    = ~np.isnan(weighted)
        x_valid  = bin_ctrs[valid]
        y_valid  = weighted[valid]

        ax.scatter(x_valid, y_valid,
                   color=reg['color'], s=40, alpha=0.8,
                   edgecolors='none', zorder=3, label=reg['name'])
        ax.errorbar(x_valid, y_valid, yerr=sems[valid],
                    fmt='none', ecolor=reg['color'],
                    elinewidth=0.8, capsize=2, alpha=0.4, zorder=2)

        # ── regression ───────────────────────────────────────────
        x_fit   = np.linspace(x_valid.min(), x_valid.max(), 200)
        fit_lbl = reg['name']
        y_fit   = None

        if REGRESSION == 'linear' and len(x_valid) >= 2:
            slope, intercept, r, p, _ = linregress(x_valid, y_valid)
            y_fit   = slope * x_fit + intercept
            fit_lbl += f'  r²={r**2:.2f}  p={p:.3f}'

        elif REGRESSION == 'poly' and len(x_valid) >= POLY_DEGREE + 1:
            coeffs  = np.polyfit(x_valid, y_valid, deg=POLY_DEGREE)
            y_fit   = np.polyval(coeffs, x_fit)
            y_pred  = np.polyval(coeffs, x_valid)
            r2      = 1 - np.sum((y_valid-y_pred)**2) / \
                          np.sum((y_valid-y_valid.mean())**2)
            fit_lbl += f'  r²={r2:.2f}  (deg {POLY_DEGREE})'

        elif REGRESSION == 'exp' and len(x_valid) >= 3:
            from scipy.optimize import curve_fit
            def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
            try:
                popt, _ = curve_fit(exp_decay, x_valid, y_valid,
                                    p0=[y_valid.max(), 0.1, y_valid.min()],
                                    maxfev=5000)
                y_fit   = exp_decay(x_fit, *popt)
                y_pred  = exp_decay(x_valid, *popt)
                r2      = 1 - np.sum((y_valid-y_pred)**2) / \
                              np.sum((y_valid-y_valid.mean())**2)
                fit_lbl += f'  r²={r2:.2f}  (exp)'
            except RuntimeError:
                print(f'Exp fit failed for {reg["name"]}')

        elif REGRESSION == 'log' and len(x_valid) >= 2:
            coeffs  = np.polyfit(np.log(x_valid + 1), y_valid, deg=1)
            y_fit   = coeffs[0] * np.log(x_fit + 1) + coeffs[1]
            y_pred  = coeffs[0] * np.log(x_valid + 1) + coeffs[1]
            r2      = 1 - np.sum((y_valid-y_pred)**2) / \
                          np.sum((y_valid-y_valid.mean())**2)
            fit_lbl += f'  r²={r2:.2f}  (log)'

        elif REGRESSION == 'lowess' and len(x_valid) >= 3:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            smoothed     = lowess(y_valid, x_valid, frac=LOWESS_FRAC)
            x_fit, y_fit = smoothed[:, 0], smoothed[:, 1]
            fit_lbl     += '  (lowess)'

        if y_fit is not None:
            ax.plot(x_fit, y_fit,
                    color=reg['color'], lw=2, ls='--',
                    zorder=4, label=fit_lbl)

    ax.set_xlabel('Distance from boundary (pixels)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Prevalence-Weighted Amplitude',   fontsize=FONT_SIZES['axis_label'])
    # ax.set_ylim(bottom=0)
    ax.set_yscale('log')
    # ax.legend(fontsize=FONT_SIZES['legend'])
    ax.spines[['top', 'right']].set_visible(False)

    if target == 'class' and class_name:
        ax.set_title(f'Prevalence-Weighted Amplitude vs Distance — {class_name}',
                     fontsize=FONT_SIZES['subtitle'])
    elif target == 'peak' and peak_center:
        ax.set_title(f'Prevalence-Weighted Amplitude vs Distance — {peak_center:.0f} cm⁻¹',
                     fontsize=FONT_SIZES['subtitle'])
    else:
        ax.set_title('Prevalence-Weighted Amplitude vs Distance',
                     fontsize=FONT_SIZES['subtitle'])

    plt.tight_layout()
    plt.show()


# for cls_name in PEAK_CLASSES:
#     plot_intensity_vs_boundary_distance(
#         regions, target='class', class_name=cls_name)

# TARGET_PEAKS = [906.85, 1442.56, 1462.37]
# for wn in TARGET_PEAKS:
#     plot_intensity_vs_boundary_distance(
#         regions, target='peak', peak_center=wn,
#         match_threshold=25.0, min_amp=0.005)

In [ ]:
plot_intensity_vs_boundary_distance(regions, target='peak', peak_center=907, figsize = (8,4))

In [ ]:
TARGET_PEAKS = [906.85, 1442.56, 1462.37]
for wn in TARGET_PEAKS:
    plot_intensity_vs_boundary_distance(
        regions, target='peak', peak_center=wn,
        match_threshold=5.0, min_amp=0.005,
        figsize=(8, 4))

In [ ]:
for wn in TARGET_PEAKS:
    print(f'\n--- looking for {wn} cm⁻¹ ---')
    for reg in regions:
        matches = [(pk['mean_center'], pk['mean_amp'])
                   for pk in reg['consensus']
                   if abs(pk['mean_center'] - wn) < 50]  # wide search
        print(f'{reg["name"]}: {matches}')

In [ ]:
def plot_spatial_sig_peaks(regions, mode='amplitude', figsize_per_map=(3, 3)):
    def _px_amps(reg, center_wn):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return sa[m] if m.any() else np.zeros(5)

    def _prev_vals(reg, center_wn):
        sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
        m = (sa > AMP_THRESHOLD) & \
            (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        return m.any(axis=2)[poly_mask_2d].astype(float)

    def _make_amp_map(reg, ref_wn, mode):
        verts = reg['verts']
        c0, r0 = verts.min(axis=0).astype(int)
        c1, r1 = verts.max(axis=0).astype(int)
        c0, r0 = max(0, c0), max(0, r0)
        c1, r1 = min(W-1, c1), min(H-1, r1)
        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
        rH, rW = sa.shape[:2]
        grid_cols, grid_rows = np.meshgrid(
            np.arange(c0, c1+1), np.arange(r0, r1+1))
        grid_pts     = np.stack([grid_cols.ravel(), grid_rows.ravel()], axis=1)
        poly_mask_2d = Path(verts).contains_points(grid_pts).reshape(rH, rW)
        poly_mask    = poly_mask_2d[:, :, np.newaxis]
        mask = (sa > AMP_THRESHOLD) & \
               (np.abs(sc - ref_wn) < CENTER_MATCH_THRESHOLD) & poly_mask
        if mode == 'amplitude':
            amp_map = (sa * mask).max(axis=2).astype(float)
        else:
            amp_map = mask.any(axis=2).astype(float)
        amp_map[~poly_mask_2d] = np.nan
        return amp_map

    ref = regions[0]
    rows = []   # each entry: dict with all info needed to plot one row

    for other in regions[1:]:
        matched_ref_idx   = {m['idx_a'] for m in other.get('matches_vs_ref', [])}
        matched_other_idx = {m['idx_b'] for m in other.get('matches_vs_ref', [])}

        # ── matched pairs — test for significant difference ───────────────
        for m in other.get('matches_vs_ref', []):
            if mode == 'amplitude':
                d1 = _px_amps(ref,   m['center_a'])
                d2 = _px_amps(other, m['center_b'])
            else:
                d1 = _prev_vals(ref,   m['center_a'])
                d2 = _prev_vals(other, m['center_b'])

            pairs = _pairwise_sig([d1, d2])
            if not pairs:
                continue
            rows.append(dict(
                kind     = 'matched',
                lbl      = pairs[0][2],
                ref_wn   = (m['center_a'] + m['center_b']) / 2,
                center_a = m['center_a'],
                center_b = m['center_b'],
                reg_b    = other,
                note     = f'{m["center_a"]:.0f} cm⁻¹ (matched) {pairs[0][2]}',
            ))

        # ── unmatched peaks in reference — absent in this other region ────
        for i, pk in enumerate(ref['consensus']):
            if i in matched_ref_idx:
                continue
            if pk.get(mode.replace('amplitude', 'mean_amp')
                         .replace('prevalence', 'prevalence'), 0) < MIN_AMP_DISPLAY:
                continue
            rows.append(dict(
                kind     = 'unmatched_ref',
                lbl      = 'absent',
                ref_wn   = pk['mean_center'],
                center_a = pk['mean_center'],
                center_b = None,
                reg_b    = other,
                note     = f'{pk["mean_center"]:.0f} cm⁻¹  only in {ref["name"]}',
            ))

        # ── unmatched peaks in other — absent in reference ────────────────
        for i, pk in enumerate(other['consensus']):
            if i in matched_other_idx:
                continue
            if pk.get(mode.replace('amplitude', 'mean_amp')
                         .replace('prevalence', 'prevalence'), 0) < MIN_AMP_DISPLAY:
                continue
            rows.append(dict(
                kind     = 'unmatched_other',
                lbl      = 'absent',
                ref_wn   = pk['mean_center'],
                center_a = None,
                center_b = pk['mean_center'],
                reg_b    = other,
                note     = f'{pk["mean_center"]:.0f} cm⁻¹  only in {other["name"]}',
            ))

    if not rows:
        print(f'Nothing to plot for mode={mode}.')
        return

    n_reg   = len(regions)
    n_rows  = len(rows)
    fig, axes = plt.subplots(
        n_rows, n_reg,
        figsize=(figsize_per_map[0] * n_reg,
                 figsize_per_map[1] * n_rows),
        squeeze=False)

    for row_i, sp in enumerate(rows):
        ref_wn = sp['ref_wn']
        maps   = []
        for reg in regions:
            # for unmatched peaks absent in this region, show blank nan map
            if sp['kind'] == 'unmatched_other' and reg is ref:
                # ref doesn't have this peak — show blank
                sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
                blank = np.full(sa.shape[:2], np.nan)
                maps.append(blank)
            elif sp['kind'] == 'unmatched_ref' and reg is sp['reg_b']:
                # other doesn't have this peak — show blank
                sa, sc, poly_mask, poly_mask_2d = _poly_slices(reg)
                blank = np.full(sa.shape[:2], np.nan)
                maps.append(blank)
            else:
                maps.append(_make_amp_map(reg, ref_wn, mode))

        vmin = np.nanmin([np.nanmin(m) for m in maps if not np.all(np.isnan(m))])
        vmax = np.nanmax([np.nanmax(m) for m in maps if not np.all(np.isnan(m))])

        for col, (reg, amp_map) in enumerate(zip(regions, maps)):
            ax  = axes[row_i][col]
            im  = ax.imshow(amp_map, cmap=plasma_rp,
                            vmin=vmin, vmax=vmax, aspect='auto')
            plt.colorbar(im, ax=ax, shrink=0.8,
                         label='Amplitude' if mode == 'amplitude' else 'Present')

            # title: color red if this region is the absent one
            is_absent = (sp['kind'] == 'unmatched_other' and reg is ref) or \
                        (sp['kind'] == 'unmatched_ref'   and reg is sp['reg_b'])
            title_sfx = '  [absent]' if is_absent else ''
            ax.set_title(reg['name'] + title_sfx,
                         color='red' if is_absent else reg['color'],
                         fontweight='bold', fontsize=9)
            ax.set_xlabel('Col', fontsize=8)
            ax.tick_params(labelsize=7)

        axes[row_i][0].set_ylabel(f'{sp["note"]}\nRow', fontsize=8, fontweight='bold')

    mode_lbl = 'Amplitude' if mode == 'amplitude' else 'Prevalence'
    fig.suptitle(
        f'Spatial Heatmaps — Significant & Unmatched Peaks ({mode_lbl})',
        fontsize=12)
    plt.tight_layout()
    plt.show()

    n_matched   = sum(1 for r in rows if r['kind'] == 'matched')
    n_unmatched = sum(1 for r in rows if r['kind'] != 'matched')
    print(f'{n_matched} significantly different matched pairs, '
          f'{n_unmatched} unmatched (absent) peaks.')


plot_spatial_sig_peaks(regions, mode='amplitude')
plot_spatial_sig_peaks(regions, mode='prevalence')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 7 — Radar / spider chart
#  One axis per class; one polygon per region
#  Good for quickly seeing which region is dominant in which class
# ═══════════════════════════════════════════════════════════════════

def plot_radar(regions, metric='mean_amp', figsize=(7, 7)):
    cls_list = list(PEAK_CLASSES.keys())
    N = len(cls_list)
    if N < 3:
        print('Radar chart needs at least 3 classes — add more to PEAK_CLASSES.')
        return
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    closed = angles + angles[:1]

    all_vals = [reg['class_summary'][cls][metric]
                for reg in regions for cls in cls_list]
    scale = max(all_vals) if max(all_vals) > 0 else 1.0

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
    for reg in regions:
        vals = [reg['class_summary'][cls][metric] / scale for cls in cls_list]
        vals_c = vals + vals[:1]
        ax.plot(closed, vals_c, color=reg['color'], lw=2, label=reg['name'])
        ax.fill(closed, vals_c, color=reg['color'], alpha=0.12)

    ax.set_xticks(angles)
    ax.set_xticklabels(cls_list, fontsize=FONT_SIZES['tick_label'])
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'],
                       fontsize=FONT_SIZES['annotation'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
              fontsize=FONT_SIZES['legend'])
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_title(f'Radar — {metric_lbl}\n(normalised to max across regions)',
                 fontsize=FONT_SIZES['title'], pad=20)
    plt.tight_layout()
    plt.show()

plot_radar(regions, metric='mean_amp')
plot_radar(regions, metric='prevalence')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 8 — Grouped bar chart with error bars  (class level)
#  Error bars = std of per-pixel amplitude within the class
#  Asterisks (*/**/***)  mark significantly different region pairs
#  (Mann-Whitney U, p < 0.05/0.01/0.001)
# ═══════════════════════════════════════════════════════════════════

def plot_bar_classes(regions, metric='mean_amp', figsize=(11, 5)):
    cls_list = list(PEAK_CLASSES.keys())
    n_cls = len(cls_list)
    n_reg = len(regions)
    x     = np.arange(n_cls)
    width = 0.8 / n_reg
    offs  = np.linspace(-0.4 + width / 2, 0.4 - width / 2, n_reg)

    raw_data = {}
    for i, reg in enumerate(regions):
        sa, sc, poly_mask, _ = _poly_slices(reg)
        for cls in cls_list:
            mask = _class_wn_mask(sa, sc, PEAK_CLASSES[cls]) & poly_mask
            raw_data[(cls, i)] = sa[mask] if mask.any() else np.array([0.0])

    fig, ax = plt.subplots(figsize=figsize)
    for reg, off in zip(regions, offs):
        vals = [reg['class_summary'][cls][metric] for cls in cls_list]
        errs = [reg['class_summary'][cls]['std_amp']  for cls in cls_list]
        ax.bar(x + off, vals, width=width, color=reg['color'], alpha=0.75,
               label=reg['name'], yerr=errs, capsize=3,
               error_kw=dict(elinewidth=1, ecolor='dimgray'))

    ax.set_xticks(x)
    ax.set_xticklabels(cls_list, rotation=15, ha='right',
                       fontsize=FONT_SIZES['tick_label'])
    ax.tick_params(axis='y', labelsize=FONT_SIZES['tick_label'])
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_ylabel(metric_lbl, fontsize=FONT_SIZES['axis_label'])
    ax.set_title(f'Class-Level {metric_lbl} per Region',
                 fontsize=FONT_SIZES['subtitle'])
    ax.legend(fontsize=FONT_SIZES['legend'])
    ax.spines[['top', 'right']].set_visible(False)

    all_tops = [
        regions[r]['class_summary'][cls][metric] + regions[r]['class_summary'][cls]['std_amp']
        for cls in cls_list for r in range(n_reg)
    ]
    dy_global = max(all_tops) * 0.07 if any(v > 0 for v in all_tops) else 0.01

    for k, cls in enumerate(cls_list):
        cls_data = [raw_data[(cls, i)] for i in range(n_reg)]
        pairs = _pairwise_sig(cls_data)
        if pairs:
            x_positions = [float(x[k]) + offs[r] for r in range(n_reg)]
            y_top = max(
                regions[r]['class_summary'][cls][metric] +
                regions[r]['class_summary'][cls]['std_amp']
                for r in range(n_reg)
            ) * 1.05
            _draw_sig_brackets(ax, pairs, x_positions, y_top, dy=dy_global,
                               fontsize=FONT_SIZES['annotation'])

    plt.tight_layout()
    plt.show()

plot_bar_classes(regions, metric='mean_amp')
plot_bar_classes(regions, metric='prevalence')
plot_bar_classes(regions, metric='mean_amp')
plot_bar_classes(regions, metric='prevalence')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 9 — Prevalence lollipop
#  Stem height = mean amplitude; head size = fraction of pixels
#  that contain that peak; head colour = peak class
# ═══════════════════════════════════════════════════════════════════

def plot_prevalence_lollipop(regions, figsize=(12, 5)):
    palette = _cls_palette()
    n      = len(regions)
    jitter = np.linspace(-3.0, 3.0, n) if n > 1 else [0.0]

    fig, ax = plt.subplots(figsize=figsize)
    for reg, jit in zip(regions, jitter):
        for pk in reg['consensus']:
            x      = pk['mean_center'] + jit
            y      = pk['mean_amp']
            sz     = 20 + 200 * pk['prevalence']   # size ∝ prevalence
            color  = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.2, alpha=0.55)
            ax.scatter(x, y, s=sz, color=color, zorder=5,
                       edgecolors=reg['color'], linewidths=0.8, alpha=0.85)

    reg_handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    leg1 = ax.legend(handles=reg_handles, loc='upper left',
                     fontsize=FONT_SIZES['legend'], title='Region')
    ax.add_artist(leg1)
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    ax.legend(handles=cls_handles, loc='upper right',
              fontsize=FONT_SIZES['legend'], title='Class')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=FONT_SIZES['axis_label'])
    ax.set_ylabel('Mean Amplitude', fontsize=FONT_SIZES['axis_label'])
    ax.set_title('Prevalence Lollipop  —  head size ∝ fraction of pixels with peak',
                 fontsize=FONT_SIZES['subtitle'])
    ax.tick_params(labelsize=FONT_SIZES['tick_label'])
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(0.99, 0.01, '● common   ○ rare',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=FONT_SIZES['annotation'], color='gray')
    plt.tight_layout()
    plt.show()

plot_prevalence_lollipop(regions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Text summary table
# ═══════════════════════════════════════════════════════════════════

def print_summary_table(regions, col_w=15):
    ref     = regions[0]
    ref_pks = ref['consensus']
    sep_w   = 36 + col_w * len(regions)

    print('=' * sep_w)
    print('MATCHED PEAK SUMMARY  (mean_amp, relative to reference region)')
    print('=' * sep_w)
    hdr = 'Center (cm⁻¹)  Class'.ljust(36)
    hdr += ''.join(r['name'].rjust(col_w) for r in regions)
    print(hdr)
    print('-' * sep_w)

    match_maps = [{}] + [{m['idx_a']: m for m in r.get('matches_vs_ref', [])}
                         for r in regions[1:]]
    for i, pk in enumerate(ref_pks):
        cls_str = str(pk.get('cls') or '-')
        row = f'{pk["mean_center"]:>8.1f}  {cls_str:<26}'
        row += f'{pk["mean_amp"]:>{col_w}.4f}'
        for mm in match_maps[1:]:
            if i in mm:
                amp_v = mm[i]['peak_b']['mean_amp']
                row  += f'{amp_v:>{col_w}.4f}'
            else:
                row  += '(absent)'.rjust(col_w)
        print(row)

    print()
    print('=' * sep_w)
    print('CLASS SUMMARY  (mean_amp | prevalence)')
    print('=' * sep_w)
    hdr2 = 'Class'.ljust(36)
    hdr2 += ''.join(r['name'].rjust(col_w) for r in regions)
    print(hdr2)
    print('-' * sep_w)
    for cls in PEAK_CLASSES:
        row2 = cls.ljust(36)
        for reg in regions:
            v     = reg['class_summary'][cls]
            entry = f'{v["mean_amp"]:.3f}/{v["prevalence"]:.1%}'
            row2 += entry.rjust(col_w)
        print(row2)
    print()

print_summary_table(regions)